# Production unlearning annotation pipeline v2

This version adds a **mandatory extraction-verification stage before any paid model calls**. It reconstructs logical paragraphs across columns and pages, removes headers, sidebars, pull quotes, references, and other non-body material, and independently verifies the result with both **PyMuPDF** and **pdfplumber**.

Workflow:
1. Upload the approved codebook/examples workbook and one or more source PDFs.
2. Run through the extraction section.
3. Inspect the generated blue extraction-audit PDFs and quality workbook.
4. Set `EXTRACTION_APPROVED = True`.
5. Run the three model coders, reliability, consensus, and final editable annotations.

The notebook will not call paid APIs unless all automated extraction checks pass **and** extraction is explicitly approved.

## 1. Install dependencies

In [1]:
%pip install -q -U openai anthropic google-genai pymupdf pdfplumber pypdf xlsxwriter pydantic python-dotenv tqdm statsmodels rapidfuzz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 998.3/998.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/

## 2. Imports and shared utilities

In [2]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from itertools import combinations
from typing import Literal, Optional
from getpass import getpass

import os
import re
import json
import time
import math
import hashlib
import shutil
import warnings

import numpy as np
import pandas as pd
import fitz  # PyMuPDF
import pdfplumber
from difflib import SequenceMatcher

from pydantic import BaseModel, ConfigDict, Field, ValidationError
from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa

try:
    from rapidfuzz.fuzz import ratio as rapid_ratio
except Exception:
    from difflib import SequenceMatcher
    def rapid_ratio(a, b):
        return 100.0 * SequenceMatcher(None, a, b).ratio()

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 100)


def env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on"}


def clean_text(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    text = str(value).replace("\u00ad", "")
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_for_match(value) -> str:
    text = clean_text(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def sha256_text(value: str) -> str:
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def safe_slug(value: str, max_len: int = 70) -> str:
    slug = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_")
    return (slug[:max_len] or "document")


def normalize_yes_no(value):
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {"yes", "true", "1", "y"}:
        return True
    if text in {"no", "false", "0", "n"}:
        return False
    return np.nan

print("Imports loaded.")

Imports loaded.


## 3. Project paths and run configuration

In [10]:
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
DEFAULT_BASE = Path("/content/unlearning_pipeline") if IN_COLAB else Path.cwd() / "unlearning_pipeline"
BASE_DIR = Path(os.getenv("UNLEARNING_BASE_DIR", str(DEFAULT_BASE))).expanduser().resolve()

INPUT_DIR = BASE_DIR / "input_documents"
OUTPUT_DIR = BASE_DIR / "outputs"
RAW_DIR = OUTPUT_DIR / "raw_provider_results"
ANNOTATED_DIR = OUTPUT_DIR / "annotated_pdfs"
EXTRACTION_AUDIT_DIR = OUTPUT_DIR / "extraction_audits"

for folder in [BASE_DIR, INPUT_DIR, OUTPUT_DIR, RAW_DIR, ANNOTATED_DIR, EXTRACTION_AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CODEBOOK_WORKBOOK_PATH = Path(os.getenv(
    "UNLEARNING_CODEBOOK_PATH",
    str(BASE_DIR / "Unlearning_Codebook_Combined_Human_Test_Set.xlsx"),
)).expanduser().resolve()
EXAMPLES_WORKBOOK_PATH = Path(os.getenv(
    "UNLEARNING_EXAMPLES_PATH",
    str(CODEBOOK_WORKBOOK_PATH),
)).expanduser().resolve()
CODEBOOK_SHEET = os.getenv("UNLEARNING_CODEBOOK_SHEET", "Codebook")
EXAMPLES_SHEET = os.getenv("UNLEARNING_EXAMPLES_SHEET", "GPT Test")
PDF_PATTERN = os.getenv("UNLEARNING_PDF_PATTERN", "*.pdf")

# A new version prevents reuse of old JSONL results created from incorrectly split blocks.
PROMPT_VERSION = "production_v2_verified_logical_paragraphs"
RUN_API_CALLS = env_bool("UNLEARNING_RUN_API_CALLS", True)
MOCK_MODE = env_bool("UNLEARNING_MOCK_MODE", False)
RESUME_FROM_JSONL = env_bool("UNLEARNING_RESUME", True)

# Mandatory human gate after viewing the extraction-audit PDFs.
EXTRACTION_APPROVED = env_bool("UNLEARNING_EXTRACTION_APPROVED", True)
FAIL_ON_EXTRACTION_CHECKS = env_bool("UNLEARNING_FAIL_ON_EXTRACTION_CHECKS", True)
EXTRACTION_ENGINE = os.getenv("UNLEARNING_EXTRACTION_ENGINE", "auto")  # auto|pymupdf|pdfplumber
STRIP_EXISTING_ANNOTATIONS = env_bool("UNLEARNING_STRIP_EXISTING_ANNOTATIONS", True)

# Logical-paragraph quality thresholds.
MIN_PARAGRAPH_TOKENS = int(os.getenv("UNLEARNING_MIN_PARAGRAPH_TOKENS", "8"))
MAX_PARAGRAPH_TOKENS = int(os.getenv("UNLEARNING_MAX_PARAGRAPH_TOKENS", "450"))
CROSS_EXTRACTOR_MEDIAN_SIMILARITY = float(os.getenv("UNLEARNING_MEDIAN_EXTRACTOR_SIM", "0.94"))
CROSS_EXTRACTOR_MIN_SIMILARITY = float(os.getenv("UNLEARNING_MIN_EXTRACTOR_SIM", "0.82"))
EXCLUDE_BACK_MATTER = env_bool("UNLEARNING_EXCLUDE_BACK_MATTER", True)
INCLUDE_ABSTRACT = env_bool("UNLEARNING_INCLUDE_ABSTRACT", True)
INCLUDE_BLOCK_QUOTES = env_bool("UNLEARNING_INCLUDE_BLOCK_QUOTES", True)
EXCLUDE_EPIGRAPHS = env_bool("UNLEARNING_EXCLUDE_EPIGRAPHS", True)

# Few-shot/leakage settings.
N_FEWSHOT_EXAMPLES = int(os.getenv("UNLEARNING_N_FEWSHOT_EXAMPLES", "6"))
FEWSHOT_SEED = int(os.getenv("UNLEARNING_FEWSHOT_SEED", "42"))
LEAKAGE_SIMILARITY_THRESHOLD = float(os.getenv("UNLEARNING_LEAKAGE_THRESHOLD", "0.92"))
EXCLUDE_SAME_DOCUMENT_EXAMPLES = env_bool("UNLEARNING_EXCLUDE_SAME_DOCUMENT_EXAMPLES", True)
MIN_SAFE_FEWSHOT_EXAMPLES = int(os.getenv("UNLEARNING_MIN_SAFE_EXAMPLES", "4"))

# API execution settings.
MAX_OUTPUT_TOKENS = int(os.getenv("UNLEARNING_MAX_OUTPUT_TOKENS", "800"))
REQUEST_SLEEP_SECONDS = float(os.getenv("UNLEARNING_REQUEST_SLEEP", "0.15"))
MAX_RETRY_ATTEMPTS = int(os.getenv("UNLEARNING_MAX_RETRIES", "4"))
MAX_UNIQUE_PARAGRAPHS = None

# Annotation behavior.
ANNOTATE_ANY_POSITIVE_VOTE = True
ANNOTATE_UNANIMOUS_NO = False
MAX_COMMENT_CHARS = 6000
MIN_ANNOTATION_TEXT_SIMILARITY = float(os.getenv("UNLEARNING_MIN_ANNOTATION_TEXT_SIM", "0.80"))

MODEL_CONFIGS = [
    {
        "provider": "openai", "display_name": "OpenAI",
        "model": os.getenv("UNLEARNING_OPENAI_MODEL", "gpt-5.6-terra"),
        "strategy": "definitions_examples_no_metadata",
        "api_key_env": "OPENAI_API_KEY", "reasoning_effort": "low",
    },
    {
        "provider": "anthropic", "display_name": "Anthropic",
        "model": os.getenv("UNLEARNING_ANTHROPIC_MODEL", "claude-sonnet-5"),
        "strategy": "definitions_examples_no_metadata",
        "api_key_env": "ANTHROPIC_API_KEY", "effort": "low",
    },
    {
        "provider": "gemini", "display_name": "Google",
        "model": os.getenv("UNLEARNING_GEMINI_MODEL", "gemini-3.1-flash-lite"),
        "strategy": "direct_no_context",
        "api_key_env": "GEMINI_API_KEY", "thinking_level": "minimal",
    },
]

print("Base directory:", BASE_DIR)
print("Input PDF directory:", INPUT_DIR)
print("Codebook workbook:", CODEBOOK_WORKBOOK_PATH)
print("RUN_API_CALLS:", RUN_API_CALLS, "| MOCK_MODE:", MOCK_MODE)
print("EXTRACTION_APPROVED:", EXTRACTION_APPROVED)
pd.DataFrame(MODEL_CONFIGS)[["provider", "model", "strategy", "api_key_env"]]

Base directory: /content/unlearning_pipeline
Input PDF directory: /content/unlearning_pipeline/input_documents
Codebook workbook: /content/unlearning_pipeline/Unlearning_Codebook_Combined_Human_Test_Set.xlsx
RUN_API_CALLS: True | MOCK_MODE: False
EXTRACTION_APPROVED: True


,provider,model,strategy,api_key_env
0,openai,gpt-5.6-terra,definitions_examples_no_metadata,OPENAI_API_KEY
1,anthropic,claude-sonnet-5,definitions_examples_no_metadata,ANTHROPIC_API_KEY
2,gemini,gemini-3.1-flash-lite,direct_no_context,GEMINI_API_KEY


### Optional Colab upload helper

In [11]:
%%script true
def upload_inputs_in_colab():
    if not IN_COLAB:
        raise RuntimeError("This helper is only available in Google Colab.")
    from google.colab import files

    uploaded = files.upload()
    copied = []
    for filename, data in uploaded.items():
        suffix = Path(filename).suffix.lower()
        destination = INPUT_DIR / filename if suffix == ".pdf" else BASE_DIR / filename
        destination.write_bytes(data)
        copied.append(str(destination))
    print("Uploaded:")
    for path in copied:
        print(" -", path)
    return copied

# Example usage:
# upload_inputs_in_colab()

## 4. API keys from environment variables, `.env`, Colab Secrets, or secure input

In [12]:
load_dotenv(BASE_DIR / ".env")
load_dotenv(Path(".env"))


def ensure_api_key(env_name: str, prompt_if_missing: bool = True) -> bool:
    if os.getenv(env_name):
        return True

    # Google Colab Secrets, when available.
    try:
        from google.colab import userdata
        secret = userdata.get(env_name)
        if secret:
            os.environ[env_name] = secret
            return True
    except Exception:
        pass

    if prompt_if_missing:
        secret = getpass(f"Enter {env_name} (input hidden; press Enter to skip): ").strip()
        if secret:
            os.environ[env_name] = secret
            return True

    return False


if RUN_API_CALLS and not MOCK_MODE:
    missing = []
    for cfg in MODEL_CONFIGS:
        if not ensure_api_key(cfg["api_key_env"], prompt_if_missing=True):
            missing.append(cfg["api_key_env"])
    if missing:
        raise RuntimeError(
            "Missing API keys: " + ", ".join(missing) +
            ". Set them in the environment, .env, Colab Secrets, or rerun this cell and enter them securely."
        )

print({cfg["api_key_env"]: bool(os.getenv(cfg["api_key_env"])) for cfg in MODEL_CONFIGS})

{'OPENAI_API_KEY': True, 'ANTHROPIC_API_KEY': True, 'GEMINI_API_KEY': True}


## 5. Load the approved codebook and labeled examples

In [14]:
EXPECTED_CODEBOOK_COLUMNS = [
    "Code",
    "Definition",
    "Detection Logic",
    "Examples",
    "Positive Clarification",
    "Negative Clarification",
]


def load_codebook(path: Path, sheet_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Codebook workbook not found: {path}")

    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    header_index = None
    for idx, row in raw.iterrows():
        values = [clean_text(x).lower() for x in row.tolist()[:8]]
        if "code" in values and "definition" in values:
            header_index = idx
            break

    if header_index is None:
        raise ValueError(
            f"Could not find a Code/Definition header row in {path.name} / {sheet_name}."
        )

    header = [clean_text(x) for x in raw.iloc[header_index].tolist()]
    df = raw.iloc[header_index + 1:].copy()
    df.columns = header
    df = df.loc[:, [c for c in EXPECTED_CODEBOOK_COLUMNS if c in df.columns]]

    for col in EXPECTED_CODEBOOK_COLUMNS:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].map(clean_text)

    df = df[df["Code"].ne("")].reset_index(drop=True)
    if df.empty:
        raise ValueError("The parsed codebook contains no coding rows.")
    return df[EXPECTED_CODEBOOK_COLUMNS]


def load_examples(path: Path, sheet_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Examples workbook not found: {path}")

    df = pd.read_excel(path, sheet_name=sheet_name)
    if "Text Content" not in df.columns:
        raise ValueError(f"{sheet_name} must contain a 'Text Content' column.")

    label_col = "Unlearning" if "Unlearning" in df.columns else "Original Unlearning"
    if label_col not in df.columns:
        raise ValueError(f"{sheet_name} must contain an Unlearning label column.")

    rename = {label_col: "example_unlearning"}
    if "Original Target" in df.columns and "Target" not in df.columns:
        rename["Original Target"] = "Target"
    if "Original Government Agency" in df.columns and "Government Agency" not in df.columns:
        rename["Original Government Agency"] = "Government Agency"
    if "Row ID" in df.columns and "Number" not in df.columns:
        rename["Row ID"] = "Number"
    if "Document Title" in df.columns and "Document" not in df.columns:
        rename["Document Title"] = "Document"

    df = df.rename(columns=rename).copy()
    df["Text Content"] = df["Text Content"].map(clean_text)
    df["example_unlearning"] = df["example_unlearning"].map(normalize_yes_no)
    df = df[df["Text Content"].ne("") & df["example_unlearning"].notna()].reset_index(drop=True)

    for col in ["Number", "Codes", "Target", "Government Agency", "Document", "Rationale"]:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].map(clean_text)

    return df


def serialize_codebook(df: pd.DataFrame) -> str:
    blocks = []
    for _, row in df.iterrows():
        blocks.append("\n".join([
            f"Code: {row['Code']}",
            f"Definition: {row['Definition']}",
            f"Detection logic: {row['Detection Logic']}",
            f"Examples: {row['Examples']}",
            f"Positive clarification: {row['Positive Clarification']}",
            f"Negative clarification: {row['Negative Clarification']}",
        ]))
    return "\n\n".join(blocks)


codebook_df = load_codebook(CODEBOOK_WORKBOOK_PATH, CODEBOOK_SHEET)
examples_pool_df = load_examples(EXAMPLES_WORKBOOK_PATH, EXAMPLES_SHEET)
CODEBOOK_TEXT = serialize_codebook(codebook_df)

print("Codebook rows:", len(codebook_df))
print("Available labeled example rows:", len(examples_pool_df))
print("Example label counts:", examples_pool_df["example_unlearning"].value_counts().to_dict())
display(codebook_df.head())

Codebook rows: 14
Available labeled example rows: 49
Example label counts: {True: 39, False: 10}


,Code,Definition,Detection Logic,Examples,Positive Clarification,Negative Clarification
0,Step 1 -- Binary classification based on unlearning definition,,,,,
1,Unlearning (Yes/No),"Unlearning is the deliberate process by which a government agency discards misaligned assumptions, obsolete knowledge, or outdated routines in response to demonstrated failure ...","Does the text explicitly identify a prior policy, assumption, practice, or technical system as inadequate or harmful — AND call for its removal, replacement, or fundamental ret...","""[T]he Federal government should not rely on the traditional layered approach and instead should proactively provide its capabilities and assistance directly to those in need.""...","Code when the text: • Names a specific prior assumption, policy, practice, or system as the source of failure. • Explicitly calls for abandoning, replacing, dismantling, or fun...",DO NOT code if: • The text only documents what went wrong without calling for change ('the levees failed'). • The change is purely additive — new resources or capacity added wi...
2,STEP 2 — TARGET OF UNLEARNING (assign one per passage),,,,,
3,Target (Leadership),"The text questions or redefines the role, authority, or responsibilities of a specific leadership position (e.g., DHS Secretary, FEMA Director, POTUS) in disaster management — ...",Does the text call for redefining or redistributing leadership authority — not just replacing the person in the role?,"""[A] single individual directly responsible and accountable to the President must be designated to act as the central focal point to lead and coordinate the overall federal res...","Code when the text: • Calls into question whether an existing leadership structure is capable of managing catastrophic events. • Proposes redistributing, consolidating, or rede...","DO NOT code if: • The text recommends appointing a new person to an existing, unchanged role. • Minor reporting-line adjustments are proposed without questioning the underlying..."
4,"Target (laws, plans and policies)","The text calls for terminating, replacing, or fundamentally revising a law, regulation, plan, or formal policy that is now seen as harmful, unjust, or structurally inadequate.","Does the text identify a specific law, plan, or policy as harmful or obsolete — and call for its elimination or replacement?","""Removing statutory restrictions on DOD authority to activate Reserve units for catastrophic disaster relief."" — GAO (Existing statutory constraint identified as harmful; remov...","Code when the text: • Calls for eliminating or replacing a specific statute, executive order, plan, or doctrine. • Frames an existing legal or policy arrangement as a root caus...","DO NOT code if: • The text proposes reform or better implementation of an existing policy without questioning its underlying logic. • A new plan is added alongside an old one, ..."


## 6. Verified logical-paragraph extraction

The extractor works at the line level rather than treating PDF text blocks as paragraphs. It:
- detects one- and two-column layouts;
- joins continuations across columns and pages;
- attaches drop caps;
- excludes repeated headers/footers, vertical licensing text, author sidebars, pull quotes, epigraphs, and references;
- reconstructs the same paragraphs independently with PyMuPDF and pdfplumber;
- validates paragraph counts, text similarity, line coverage, geometry, suspicious starts/endings, and paragraph lengths;
- creates a blue extraction-audit PDF for every source document.

In [15]:
from __future__ import annotations

from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
from difflib import SequenceMatcher
import hashlib
import json
import math
import re

import fitz
import numpy as np
import pandas as pd
import pdfplumber

LIGATURE_MAP = str.maketrans({
    "\ufb00": "ff", "\ufb01": "fi", "\ufb02": "fl",
    "\ufb03": "ffi", "\ufb04": "ffl", "\ufb05": "st", "\ufb06": "st",
})
TERMINAL_RE = re.compile(r'[.!?][\"”’\')\]]*(?:\s*\([^)]*\))?$')
CONTINUATION_WORDS = set(
    "and or but because while although however moreover furthermore therefore thus than "
    "which who that where when with without to of for from in on at as by into onto upon "
    "through toward towards among between".split()
)
BACK_MATTER_HEADINGS = {
    "references", "bibliography", "works cited", "literature cited", "notes", "endnotes"
}


@dataclass
class ExtractionConfig:
    engine: str = "auto"  # auto | pymupdf | pdfplumber
    min_tokens: int = 8
    max_tokens: int = 450
    paragraph_gap_points: float = 6.0
    repeated_margin_fraction: float = 0.40
    cross_extractor_min_median_similarity: float = 0.94
    cross_extractor_min_paragraph_similarity: float = 0.82
    max_unmatched_paragraphs: int = 1
    exclude_back_matter: bool = True
    include_abstract: bool = True
    include_block_quotes: bool = True
    exclude_epigraphs: bool = True


def normalize_unicode(text: str) -> str:
    text = (text or "").translate(LIGATURE_MAP)
    return (
        text.replace("\u00ad", "")
        .replace("\u2010", "-")
        .replace("\u2011", "-")
        .replace("\u2212", "-")
    )


def clean_line(text: str) -> str:
    text = normalize_unicode(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_joined_text(text: str) -> str:
    text = clean_line(text)
    # Common artifacts from older scholarly PDFs.
    text = re.sub(r"\bTh\s+(?=[a-z])", "Th", text)
    text = re.sub(r"\bth\s+(?=[a-z])", "th", text)
    repairs = {
        "fi eld": "field", "fi rst": "first", "fi nal": "final", "fi scal": "fiscal",
        "fi nd": "find", "fi lled": "filled", "fi ve": "five", "fi x": "fix",
        "fi xes": "fixes", "fi nalized": "finalized", "fi nancial": "financial",
        "eff ect": "effect", "eff ective": "effective", "diff erent": "different",
        "off er": "offer", "off ered": "offered", "refl ect": "reflect",
        "fl exibility": "flexibility", "confl ict": "conflict", "signifi cant": "significant",
        "specifi c": "specific", "identifi ed": "identified", "justifi ably": "justifiably",
        "nonprofi t": "nonprofit", "profi t": "profit", "ineff ective": "ineffective",
    }
    for bad, good in repairs.items():
        text = re.sub(rf"\b{re.escape(bad)}\b", good, text, flags=re.I)
    return text


def normalize_for_match(text: str) -> str:
    text = clean_joined_text(text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def token_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", str(text), flags=re.UNICODE))


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def safe_slug(value: str, max_len: int = 80) -> str:
    slug = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_")
    return (slug[:max_len] or "document")


def extract_lines_pymupdf(pdf_path: Path) -> pd.DataFrame:
    rows = []
    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            page_dict = page.get_text("dict", sort=False)
            for block_index, block in enumerate(page_dict.get("blocks", [])):
                if block.get("type", 0) != 0:
                    continue
                for line_index, line in enumerate(block.get("lines", [])):
                    spans = [s for s in line.get("spans", []) if clean_line(s.get("text", ""))]
                    if not spans:
                        continue
                    text = clean_line("".join(s.get("text", "") for s in spans))
                    bbox = line.get("bbox")
                    weights = np.array([max(len(s.get("text", "")), 1) for s in spans])
                    sizes = np.array([float(s.get("size", 0)) for s in spans])
                    fonts = [str(s.get("font", "")) for s in spans]
                    direction = line.get("dir", (1.0, 0.0))
                    rows.append({
                        "engine": "pymupdf",
                        "page_number": page_number,
                        "line_id": f"f_{page_number}_{block_index}_{line_index}",
                        "text": text,
                        "x0": float(bbox[0]), "y0": float(bbox[1]),
                        "x1": float(bbox[2]), "y1": float(bbox[3]),
                        "font_size": float(np.average(sizes, weights=weights)),
                        "bold": any(re.search(r"bold|black|semi", f, re.I) for f in fonts),
                        "italic": any(re.search(r"italic|oblique", f, re.I) for f in fonts),
                        "dir_x": float(direction[0]), "dir_y": float(direction[1]),
                        "fonts": "|".join(sorted(set(fonts))),
                    })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["width"] = df["x1"] - df["x0"]
    df["height"] = df["y1"] - df["y0"]
    return df


def extract_lines_pdfplumber(pdf_path: Path, y_tolerance: float = 2.0, x_gap: float = 9.0) -> pd.DataFrame:
    rows = []
    with pdfplumber.open(pdf_path) as doc:
        for page_number, page in enumerate(doc.pages, start=1):
            words = page.extract_words(
                x_tolerance=1.5,
                y_tolerance=2,
                keep_blank_chars=False,
                use_text_flow=False,
                extra_attrs=["fontname", "size"],
            )
            words = sorted(words, key=lambda w: (w["top"], w["x0"]))
            y_clusters = []
            for word in words:
                center_y = (word["top"] + word["bottom"]) / 2
                target = None
                for cluster in reversed(y_clusters[-8:]):
                    if abs(center_y - cluster["center_y"]) <= y_tolerance:
                        target = cluster
                        break
                if target is None:
                    y_clusters.append({"center_y": center_y, "words": [word]})
                else:
                    target["words"].append(word)
                    target["center_y"] = float(np.median([
                        (x["top"] + x["bottom"]) / 2 for x in target["words"]
                    ]))

            line_index = 0
            for cluster in y_clusters:
                ordered_words = sorted(cluster["words"], key=lambda w: w["x0"])
                groups, current, previous_x1 = [], [], None
                for word in ordered_words:
                    if current and word["x0"] - previous_x1 > x_gap:
                        groups.append(current)
                        current = []
                    current.append(word)
                    previous_x1 = word["x1"]
                if current:
                    groups.append(current)

                for group in groups:
                    text = clean_line(" ".join(w["text"] for w in group))
                    if not text:
                        continue
                    weights = np.array([max(len(w["text"]), 1) for w in group])
                    sizes = np.array([float(w.get("size") or 0) for w in group])
                    fonts = [str(w.get("fontname") or "") for w in group]
                    rows.append({
                        "engine": "pdfplumber",
                        "page_number": page_number,
                        "line_id": f"p_{page_number}_{line_index}",
                        "text": text,
                        "x0": float(min(w["x0"] for w in group)),
                        "y0": float(min(w["top"] for w in group)),
                        "x1": float(max(w["x1"] for w in group)),
                        "y1": float(max(w["bottom"] for w in group)),
                        "font_size": float(np.average(sizes, weights=weights)) if len(sizes) else 0.0,
                        "bold": any(re.search(r"bold|black|semi", f, re.I) for f in fonts),
                        "italic": any(re.search(r"italic|oblique", f, re.I) for f in fonts),
                        "dir_x": 1.0, "dir_y": 0.0,
                        "fonts": "|".join(sorted(set(fonts))),
                    })
                    line_index += 1
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["width"] = df["x1"] - df["x0"]
    df["height"] = df["y1"] - df["y0"]
    return df


def estimate_body_font_size(lines: pd.DataFrame) -> float:
    candidates = lines[
        (lines["dir_y"].abs() < 0.15)
        & lines["font_size"].between(6.5, 14)
        & (lines["text"].str.len() > 20)
    ]
    if candidates.empty:
        return 10.0
    rounded = (candidates["font_size"] * 2).round() / 2
    weighted = Counter()
    for value, weight in zip(rounded, candidates["text"].str.len()):
        weighted[float(value)] += int(weight)
    return float(weighted.most_common(1)[0][0])


def page_dimensions(pdf_path: Path) -> dict[int, tuple[float, float]]:
    with fitz.open(pdf_path) as doc:
        return {i + 1: (float(page.rect.width), float(page.rect.height)) for i, page in enumerate(doc)}


def repeated_margin_signatures(
    lines: pd.DataFrame,
    dimensions: dict[int, tuple[float, float]],
    page_fraction: float,
) -> set[str]:
    pages_by_signature = defaultdict(set)
    for row in lines.itertuples():
        width, height = dimensions[row.page_number]
        horizontal = abs(row.dir_y) < 0.15 and row.dir_x > 0.8
        marginal = (
            row.y1 < height * 0.10
            or row.y0 > height * 0.90
            or row.x0 > width * 0.94
            or row.x1 < width * 0.06
            or not horizontal
        )
        if not marginal:
            continue
        signature = normalize_for_match(re.sub(r"\d+", "0", row.text))
        if 3 <= len(signature) <= 220:
            pages_by_signature[signature].add(row.page_number)
    minimum_pages = max(2, math.ceil(len(dimensions) * page_fraction))
    return {
        signature for signature, pages in pages_by_signature.items()
        if len(pages) >= minimum_pages
    }


def detect_columns(page_lines: pd.DataFrame, page_width: float, body_size: float):
    candidates = page_lines[
        (page_lines["dir_y"].abs() < 0.15)
        & page_lines["font_size"].between(body_size - 0.8, body_size + 0.8)
        & page_lines["width"].between(60, page_width * 0.58)
    ].copy()
    if len(candidates) < 8:
        return 1, [page_width / 2], 0.0
    centers_x = np.sort(((candidates["x0"] + candidates["x1"]) / 2).to_numpy())
    if len(centers_x) < 2:
        return 1, [float(np.mean(centers_x))], 0.0
    gaps = np.diff(centers_x)
    split_index = int(np.argmax(gaps))
    largest_gap = float(gaps[split_index])
    left = centers_x[: split_index + 1]
    right = centers_x[split_index + 1 :]
    separation_score = largest_gap / max(float(centers_x[-1] - centers_x[0]), 1.0)
    if (
        largest_gap > page_width * 0.14
        and len(left) >= 3
        and len(right) >= 3
        and float(np.mean(right) - np.mean(left)) > page_width * 0.24
    ):
        return 2, [float(np.mean(left)), float(np.mean(right))], separation_score
    return 1, [float(np.mean(centers_x))], separation_score


def classify_lines(lines: pd.DataFrame, pdf_path: Path, config: ExtractionConfig):
    body_size = estimate_body_font_size(lines)
    dimensions = page_dimensions(pdf_path)
    repeated = repeated_margin_signatures(lines, dimensions, config.repeated_margin_fraction)

    back_rows = lines[
        lines["text"].map(normalize_for_match).isin(BACK_MATTER_HEADINGS)
    ].sort_values(["page_number", "y0"])
    back_page = int(back_rows.iloc[0]["page_number"]) if not back_rows.empty else None
    back_y = float(back_rows.iloc[0]["y0"]) if not back_rows.empty else None

    page_columns = {}
    for page_number in sorted(lines["page_number"].unique()):
        page_columns[page_number] = detect_columns(
            lines[lines["page_number"].eq(page_number)],
            dimensions[page_number][0],
            body_size,
        )

    roles, columns, reasons = [], [], []
    for row in lines.itertuples():
        width, height = dimensions[row.page_number]
        signature = normalize_for_match(re.sub(r"\d+", "0", row.text))
        horizontal = abs(row.dir_y) < 0.15 and row.dir_x > 0.8

        if not horizontal:
            role, reason = "excluded", "rotated_or_vertical_margin"
        elif signature in repeated and (row.y1 < height * 0.12 or row.y0 > height * 0.88):
            role, reason = "excluded", "repeated_header_or_footer"
        elif config.exclude_back_matter and back_page and (
            row.page_number > back_page or (row.page_number == back_page and row.y0 >= back_y)
        ):
            role, reason = "excluded", "back_matter"
        elif row.font_size <= body_size - 1.5:
            role, reason = "excluded", "small_metadata_or_footnote"
        elif row.font_size >= body_size + 1.5:
            if re.fullmatch(r"[A-Za-z]", row.text.strip()):
                role, reason = "dropcap", "drop_cap"
            elif row.bold or row.width > width * 0.55 or token_count(row.text) <= 12:
                role, reason = "heading", "title_or_heading"
            else:
                role, reason = "excluded", "pull_quote_or_display_text"
        elif row.bold and token_count(row.text) <= 12:
            role, reason = "heading", "section_heading"
        elif re.fullmatch(r"(?:page\s*)?\d+(?:\s+of\s+\d+)?", normalize_for_match(row.text)):
            role, reason = "excluded", "page_number"
        else:
            role, reason = "body", "body_line"

        number_columns, centers, _ = page_columns[row.page_number]
        center_x = (row.x0 + row.x1) / 2
        column = -1 if row.width > width * 0.72 else int(np.argmin([abs(center_x - x) for x in centers]))
        roles.append(role)
        columns.append(column)
        reasons.append(reason)

    classified = lines.copy()
    classified["role"] = roles
    classified["column"] = columns
    classified["exclusion_reason"] = reasons

    # Attach oversized drop caps to the nearest body line.
    drop_indices = []
    for index, row in classified[classified["role"].eq("dropcap")].iterrows():
        candidates = classified[
            classified["page_number"].eq(row["page_number"])
            & classified["role"].eq("body")
            & (classified["x0"] >= row["x0"])
            & (classified["y0"] >= row["y0"] - 3)
            & (classified["y0"] <= row["y1"] + 5)
        ].copy()
        if candidates.empty:
            continue
        candidates["distance"] = (
            (candidates["y0"] - row["y0"]).abs()
            + (candidates["x0"] - row["x1"]).abs() / 10
        )
        target_index = candidates.sort_values("distance").index[0]
        classified.at[target_index, "text"] = row["text"].strip() + classified.at[target_index, "text"]
        classified.at[index, "role"] = "excluded"
        classified.at[index, "exclusion_reason"] = "drop_cap_attached"
        drop_indices.append(index)

    return classified, body_size, page_columns, dimensions


def join_line_texts(texts: list[str]) -> str:
    output = ""
    for text in texts:
        text = clean_line(text)
        if not output:
            output = text
        elif output.endswith("-") and re.match(r"^[a-z]", text):
            output = output[:-1] + text
        else:
            output += " " + text
    return clean_joined_text(output)


def _make_paragraph(lines: list, page_number: int, column: int):
    return {
        "text": join_line_texts([row.text for row in lines]),
        "start_page": int(page_number),
        "end_page": int(page_number),
        "column_start": int(column),
        "column_end": int(column),
        "section_heading": "",
        "segments": [{
            "page_number": int(page_number),
            "rects": [[row.x0, row.y0, row.x1, row.y1] for row in lines],
            "line_ids": [row.line_id for row in lines],
            "text": join_line_texts([row.text for row in lines]),
        }],
        "font_size": float(np.median([row.font_size for row in lines])),
        "italic_fraction": float(np.mean([row.italic for row in lines])),
        "bold_fraction": float(np.mean([row.bold for row in lines])),
        "line_count": len(lines),
    }


def group_body_lines(
    classified: pd.DataFrame,
    page_columns: dict,
    body_size: float,
    config: ExtractionConfig,
) -> list[dict]:
    paragraphs = []
    headings = classified[classified["role"].eq("heading")].copy()

    for page_number in sorted(classified["page_number"].unique()):
        body_lines = classified[
            classified["page_number"].eq(page_number)
            & classified["role"].eq("body")
        ].copy()
        number_columns, _, _ = page_columns[page_number]

        for column in range(number_columns):
            column_lines = body_lines[body_lines["column"].eq(column)].sort_values(["y0", "x0"])
            current = []
            for row in column_lines.itertuples():
                if current and row.y0 - current[-1].y1 > config.paragraph_gap_points:
                    paragraphs.append(_make_paragraph(current, page_number, column))
                    current = []
                current.append(row)
            if current:
                paragraphs.append(_make_paragraph(current, page_number, column))

    # Carry the closest preceding section heading in the same column.
    current_section = ""
    paragraphs = sorted(
        paragraphs,
        key=lambda p: (
            p["start_page"], p["column_start"], p["segments"][0]["rects"][0][1]
        ),
    )
    for paragraph in paragraphs:
        page_number = paragraph["start_page"]
        column = paragraph["column_start"]
        y0 = paragraph["segments"][0]["rects"][0][1]
        candidates = headings[
            headings["page_number"].eq(page_number)
            & headings["column"].eq(column)
            & (headings["y0"] < y0)
            & ((y0 - headings["y1"]) < 80)
        ].sort_values("y0")
        if not candidates.empty:
            current_section = clean_joined_text(" ".join(candidates["text"].tolist()))
        paragraph["section_heading"] = current_section
    return paragraphs


def _ends_terminal(text: str) -> bool:
    return bool(TERMINAL_RE.search(text.strip()))


def _starts_as_continuation(text: str) -> bool:
    text = text.strip()
    match = re.match(r'[\[\(\"“‘]*([A-Za-z]+)', text)
    if not match:
        return True
    word = match.group(1)
    return word[0].islower() or word.lower() in CONTINUATION_WORDS


def should_merge_boundary(previous: dict, current: dict) -> bool:
    boundary = (
        current["start_page"] > previous["end_page"]
        or (
            current["start_page"] == previous["end_page"]
            and current["column_start"] != previous["column_end"]
        )
    )
    if not boundary:
        return False
    previous_text = previous["text"].strip()
    current_text = current["text"].strip()
    if previous_text.endswith("-"):
        return True
    if not _ends_terminal(previous_text) and _starts_as_continuation(current_text):
        return True
    if previous_text.endswith(("“", '"', "'", "’")) and _starts_as_continuation(current_text):
        return True
    return False


def combine_paragraphs(previous: dict, current: dict, reason: str) -> dict:
    total_lines = previous["line_count"] + current["line_count"]
    combined = dict(previous)
    combined.update({
        "text": join_line_texts([previous["text"], current["text"]]),
        "end_page": current["end_page"],
        "column_end": current["column_end"],
        "segments": previous["segments"] + current["segments"],
        "font_size": float(np.median([previous["font_size"], current["font_size"]])),
        "italic_fraction": (
            previous["italic_fraction"] * previous["line_count"]
            + current["italic_fraction"] * current["line_count"]
        ) / total_lines,
        "bold_fraction": (
            previous["bold_fraction"] * previous["line_count"]
            + current["bold_fraction"] * current["line_count"]
        ) / total_lines,
        "line_count": total_lines,
        "section_heading": previous["section_heading"] or current["section_heading"],
        "merge_reasons": previous.get("merge_reasons", []) + [reason] + current.get("merge_reasons", []),
    })
    return combined


def merge_continuations(paragraphs: list[dict], config: ExtractionConfig) -> list[dict]:
    merged = []
    for paragraph in paragraphs:
        paragraph = dict(paragraph)
        paragraph.setdefault("merge_reasons", [])
        if merged and should_merge_boundary(merged[-1], paragraph):
            boundary_type = (
                "cross_page_continuation"
                if paragraph["start_page"] > merged[-1]["end_page"]
                else "cross_column_continuation"
            )
            merged[-1] = combine_paragraphs(merged[-1], paragraph, boundary_type)
        else:
            merged.append(paragraph)

    epigraph_indices = set()
    for index, paragraph in enumerate(merged):
        if (
            re.match(r'^[—-]\s*[A-Z].*\b(?:18|19|20)\d{2}\b', paragraph["text"])
            and index > 0
            and merged[index - 1]["italic_fraction"] > 0.80
        ):
            epigraph_indices.update({index - 1, index})

    for index, paragraph in enumerate(merged):
        if index in epigraph_indices:
            paragraph["content_type"] = "epigraph"
            paragraph["include_for_labeling"] = not config.exclude_epigraphs
        elif paragraph["italic_fraction"] > 0.80 and paragraph["start_page"] == 1:
            paragraph["content_type"] = "abstract"
            paragraph["include_for_labeling"] = config.include_abstract
        elif paragraph["text"].lstrip().startswith("["):
            paragraph["content_type"] = "block_quote"
            paragraph["include_for_labeling"] = config.include_block_quotes
        else:
            paragraph["content_type"] = "body"
            paragraph["include_for_labeling"] = True
    return merged


def paragraphs_to_dataframe(
    paragraphs: list[dict],
    pdf_path: Path,
    engine: str,
) -> pd.DataFrame:
    rows = []
    document_id = safe_slug(pdf_path.stem)
    sequence = 0
    for paragraph in paragraphs:
        if not paragraph["include_for_labeling"]:
            continue
        sequence += 1
        normalized = normalize_for_match(paragraph["text"])
        page_numbers = sorted({int(s["page_number"]) for s in paragraph["segments"]})
        rows.append({
            "source_file": str(pdf_path),
            "source_filename": pdf_path.name,
            "doc_id": document_id,
            "extraction_engine": engine,
            "paragraph_id": f"{document_id}_para_{sequence:04d}",
            "paragraph_order": sequence,
            "start_page": int(paragraph["start_page"]),
            "end_page": int(paragraph["end_page"]),
            "page_numbers": ",".join(map(str, page_numbers)),
            "spans_multiple_pages": len(page_numbers) > 1,
            "spans_multiple_columns": paragraph["column_start"] != paragraph["column_end"],
            "content_type": paragraph["content_type"],
            "section_heading": paragraph["section_heading"],
            "token_count": token_count(paragraph["text"]),
            "line_count": paragraph["line_count"],
            "text": paragraph["text"],
            "normalized_text": normalized,
            "text_hash": sha256_text(normalized),
            "segments_json": json.dumps(paragraph["segments"]),
            "merge_reasons": "; ".join(paragraph.get("merge_reasons", [])),
            "starts_lowercase": bool(re.match(r"^[a-z]", paragraph["text"].strip())),
            "ends_terminal": _ends_terminal(paragraph["text"]),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        next_type = df["content_type"].shift(-1)
        next_same_document = df["doc_id"].shift(-1).eq(df["doc_id"])
        df["dangling_nonterminal"] = (
            ~df["ends_terminal"]
            & ~(next_same_document & next_type.eq("block_quote"))
        )
    return df


def extract_with_engine(pdf_path: Path, engine: str, config: ExtractionConfig):
    if engine == "pymupdf":
        raw_lines = extract_lines_pymupdf(pdf_path)
    elif engine == "pdfplumber":
        raw_lines = extract_lines_pdfplumber(pdf_path)
    else:
        raise ValueError(f"Unsupported extraction engine: {engine}")
    if raw_lines.empty:
        raise RuntimeError(f"{engine} extracted no lines from {pdf_path.name}")
    classified, body_size, page_columns, dimensions = classify_lines(raw_lines, pdf_path, config)
    initial_paragraphs = group_body_lines(classified, page_columns, body_size, config)
    logical_paragraphs = merge_continuations(initial_paragraphs, config)
    paragraph_df = paragraphs_to_dataframe(logical_paragraphs, pdf_path, engine)
    return paragraph_df, classified, body_size, page_columns, dimensions


def align_paragraph_sets(primary: pd.DataFrame, secondary: pd.DataFrame, window: int = 3) -> pd.DataFrame:
    rows = []
    secondary_index = 0
    used = set()
    for _, primary_row in primary.iterrows():
        candidate_indices = [
            idx for idx in range(max(0, secondary_index - 1), min(len(secondary), secondary_index + window + 1))
            if idx not in used
        ]
        if not candidate_indices:
            rows.append({
                "primary_paragraph_id": primary_row["paragraph_id"],
                "secondary_paragraph_id": "",
                "similarity": 0.0,
                "primary_text": primary_row["text"],
                "secondary_text": "",
            })
            continue
        scored = []
        for idx in candidate_indices:
            secondary_row = secondary.iloc[idx]
            score = SequenceMatcher(
                None,
                primary_row["normalized_text"],
                secondary_row["normalized_text"],
            ).ratio()
            scored.append((score, idx, secondary_row))
        score, best_index, secondary_row = max(scored, key=lambda x: x[0])
        used.add(best_index)
        secondary_index = best_index + 1
        rows.append({
            "primary_paragraph_id": primary_row["paragraph_id"],
            "secondary_paragraph_id": secondary_row["paragraph_id"],
            "similarity": float(score),
            "primary_text": primary_row["text"],
            "secondary_text": secondary_row["text"],
        })
    for idx, secondary_row in secondary.iterrows():
        if idx not in used:
            rows.append({
                "primary_paragraph_id": "",
                "secondary_paragraph_id": secondary_row["paragraph_id"],
                "similarity": 0.0,
                "primary_text": "",
                "secondary_text": secondary_row["text"],
            })
    return pd.DataFrame(rows)


def page_token_agreement(pdf_path: Path, primary_lines: pd.DataFrame, secondary_lines: pd.DataFrame) -> pd.DataFrame:
    rows = []
    pages = sorted(set(primary_lines["page_number"]) | set(secondary_lines["page_number"]))
    for page_number in pages:
        primary_text = normalize_for_match(" ".join(
            primary_lines[
                primary_lines["page_number"].eq(page_number)
                & primary_lines["role"].ne("excluded")
            ]["text"].tolist()
        ))
        secondary_text = normalize_for_match(" ".join(
            secondary_lines[
                secondary_lines["page_number"].eq(page_number)
                & secondary_lines["role"].ne("excluded")
            ]["text"].tolist()
        ))
        primary_tokens = Counter(primary_text.split())
        secondary_tokens = Counter(secondary_text.split())
        intersection = sum((primary_tokens & secondary_tokens).values())
        precision = intersection / max(sum(primary_tokens.values()), 1)
        recall = intersection / max(sum(secondary_tokens.values()), 1)
        f1 = 1.0 if not primary_tokens and not secondary_tokens else 2 * precision * recall / max(precision + recall, 1e-12)
        rows.append({
            "page_number": page_number,
            "primary_token_count": sum(primary_tokens.values()),
            "secondary_token_count": sum(secondary_tokens.values()),
            "token_multiset_f1": f1,
        })
    return pd.DataFrame(rows)


def validate_geometry(paragraph_df: pd.DataFrame, pdf_path: Path) -> pd.DataFrame:
    dimensions = page_dimensions(pdf_path)
    rows = []
    for _, paragraph in paragraph_df.iterrows():
        issues = []
        line_ids = []
        for segment in json.loads(paragraph["segments_json"]):
            page_number = int(segment["page_number"])
            width, height = dimensions[page_number]
            line_ids.extend(segment.get("line_ids", []))
            for rect in segment["rects"]:
                x0, y0, x1, y1 = map(float, rect)
                if not (0 <= x0 < x1 <= width + 1 and 0 <= y0 < y1 <= height + 1):
                    issues.append(f"invalid_rect_page_{page_number}")
        if len(line_ids) != len(set(line_ids)):
            issues.append("duplicate_line_inside_paragraph")
        rows.append({
            "paragraph_id": paragraph["paragraph_id"],
            "geometry_valid": not issues,
            "geometry_issues": "; ".join(sorted(set(issues))),
        })
    return pd.DataFrame(rows)


def build_quality_checks(
    selected: pd.DataFrame,
    selected_lines: pd.DataFrame,
    alternate: pd.DataFrame,
    alignment: pd.DataFrame,
    page_agreement: pd.DataFrame,
    geometry: pd.DataFrame,
    config: ExtractionConfig,
) -> pd.DataFrame:
    similarities = alignment[alignment["similarity"].gt(0)]["similarity"]
    unmatched = int((alignment["similarity"].eq(0)).sum())
    body_line_ids = set(selected_lines[selected_lines["role"].eq("body")]["line_id"])
    assigned_line_ids = set()
    for value in selected["segments_json"]:
        for segment in json.loads(value):
            assigned_line_ids.update(segment.get("line_ids", []))
    assignment_coverage = len(body_line_ids & assigned_line_ids) / max(len(body_line_ids), 1)

    checks = [
        ("paragraph_count_nonzero", len(selected) > 0, len(selected), "> 0"),
        ("cross_extractor_count_difference", abs(len(selected) - len(alternate)) <= config.max_unmatched_paragraphs,
         abs(len(selected) - len(alternate)), f"<= {config.max_unmatched_paragraphs}"),
        ("cross_extractor_median_similarity", similarities.median() >= config.cross_extractor_min_median_similarity,
         float(similarities.median()) if not similarities.empty else 0.0,
         f">= {config.cross_extractor_min_median_similarity}"),
        ("cross_extractor_min_similarity", similarities.min() >= config.cross_extractor_min_paragraph_similarity,
         float(similarities.min()) if not similarities.empty else 0.0,
         f">= {config.cross_extractor_min_paragraph_similarity}"),
        ("unmatched_paragraphs", unmatched <= config.max_unmatched_paragraphs,
         unmatched, f"<= {config.max_unmatched_paragraphs}"),
        ("body_line_assignment_coverage", assignment_coverage >= 0.98,
         assignment_coverage, ">= 0.98 (epigraphs may be intentionally excluded)"),
        ("no_lowercase_orphan_starts", int(selected["starts_lowercase"].sum()) == 0,
         int(selected["starts_lowercase"].sum()), "0"),
        ("no_dangling_nonterminal_paragraphs", int(selected["dangling_nonterminal"].sum()) == 0,
         int(selected["dangling_nonterminal"].sum()), "0"),
        ("minimum_paragraph_length", int((selected["token_count"] < config.min_tokens).sum()) == 0,
         int((selected["token_count"] < config.min_tokens).sum()), "0"),
        ("maximum_paragraph_length", int((selected["token_count"] > config.max_tokens).sum()) == 0,
         int((selected["token_count"] > config.max_tokens).sum()), "0"),
        ("all_geometry_valid", bool(geometry["geometry_valid"].all()),
         int((~geometry["geometry_valid"]).sum()), "0 invalid"),
        ("page_text_agreement", bool((page_agreement["token_multiset_f1"] >= 0.90).all()),
         float(page_agreement["token_multiset_f1"].min()), ">= 0.90 on every page"),
    ]
    return pd.DataFrame(checks, columns=["check", "passed", "observed", "required"])


def extraction_score(paragraph_df: pd.DataFrame, quality_checks: pd.DataFrame) -> float:
    pass_rate = quality_checks["passed"].mean() if not quality_checks.empty else 0.0
    suspicious = (
        paragraph_df["starts_lowercase"].sum()
        + (paragraph_df["token_count"] < 8).sum()
        + (paragraph_df["token_count"] > 450).sum()
    )
    return float(pass_rate * 100 - suspicious * 5)


def extract_document(pdf_path: Path, config: ExtractionConfig | None = None) -> dict:
    config = config or ExtractionConfig()
    pdf_path = Path(pdf_path)
    engines = ["pymupdf", "pdfplumber"]
    results = {}
    for engine in engines:
        paragraphs, lines, body_size, columns, dimensions = extract_with_engine(pdf_path, engine, config)
        results[engine] = {
            "paragraphs": paragraphs,
            "lines": lines,
            "body_size": body_size,
            "columns": columns,
            "dimensions": dimensions,
        }

    # PyMuPDF remains preferred for annotation geometry when the candidates agree.
    primary_engine = "pymupdf" if config.engine == "auto" else config.engine
    alternate_engine = "pdfplumber" if primary_engine == "pymupdf" else "pymupdf"
    primary = results[primary_engine]
    alternate = results[alternate_engine]
    alignment = align_paragraph_sets(primary["paragraphs"], alternate["paragraphs"])
    page_agreement = page_token_agreement(pdf_path, primary["lines"], alternate["lines"])
    geometry = validate_geometry(primary["paragraphs"], pdf_path)
    checks = build_quality_checks(
        primary["paragraphs"], primary["lines"], alternate["paragraphs"],
        alignment, page_agreement, geometry, config,
    )

    if config.engine == "auto" and not checks["passed"].all():
        # Evaluate the alternate as primary, then choose the higher-quality candidate.
        alternate_alignment = align_paragraph_sets(alternate["paragraphs"], primary["paragraphs"])
        alternate_page_agreement = page_token_agreement(pdf_path, alternate["lines"], primary["lines"])
        alternate_geometry = validate_geometry(alternate["paragraphs"], pdf_path)
        alternate_checks = build_quality_checks(
            alternate["paragraphs"], alternate["lines"], primary["paragraphs"],
            alternate_alignment, alternate_page_agreement, alternate_geometry, config,
        )
        if extraction_score(alternate["paragraphs"], alternate_checks) > extraction_score(primary["paragraphs"], checks):
            primary_engine, alternate_engine = alternate_engine, primary_engine
            primary, alternate = alternate, primary
            alignment, page_agreement, geometry, checks = (
                alternate_alignment, alternate_page_agreement, alternate_geometry, alternate_checks
            )

    selected = primary["paragraphs"].merge(geometry, on="paragraph_id", how="left")
    selected["cross_extractor_similarity"] = alignment.set_index("primary_paragraph_id")["similarity"].reindex(
        selected["paragraph_id"]
    ).fillna(0).to_numpy()
    selected["manual_review_required"] = (
        (selected["cross_extractor_similarity"] < config.cross_extractor_min_paragraph_similarity)
        | (~selected["geometry_valid"])
        | selected["starts_lowercase"]
        | selected["dangling_nonterminal"]
        | (selected["token_count"] < config.min_tokens)
        | (selected["token_count"] > config.max_tokens)
    )

    return {
        "pdf_path": pdf_path,
        "selected_engine": primary_engine,
        "alternate_engine": alternate_engine,
        "paragraphs": selected,
        "line_audit": primary["lines"],
        "alternate_paragraphs": alternate["paragraphs"],
        "alignment": alignment,
        "page_agreement": page_agreement,
        "geometry": geometry,
        "quality_checks": checks,
        "all_checks_passed": bool(checks["passed"].all()),
        "body_font_size": primary["body_size"],
        "page_columns": primary["columns"],
    }


def strip_annotations(source_path: Path, output_path: Path) -> int:
    removed = 0
    with fitz.open(source_path) as doc:
        for page in doc:
            annot = page.first_annot
            while annot:
                next_annot = annot.next
                page.delete_annot(annot)
                removed += 1
                annot = next_annot
        doc.save(output_path, garbage=4, deflate=True)
    return removed


def create_extraction_audit_pdf(
    source_path: Path,
    paragraph_df: pd.DataFrame,
    line_audit: pd.DataFrame,
    output_path: Path,
    include_excluded_overlay: bool = False,
) -> dict:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    annotations_added = 0
    existing_removed = 0
    with fitz.open(source_path) as doc:
        # Remove prior model/audit annotations from the audit copy.
        for page in doc:
            annot = page.first_annot
            while annot:
                next_annot = annot.next
                page.delete_annot(annot)
                existing_removed += 1
                annot = next_annot

        for _, row in paragraph_df.iterrows():
            segments = json.loads(row["segments_json"])
            for segment_index, segment in enumerate(segments, start=1):
                page = doc[int(segment["page_number"]) - 1]
                rects = [fitz.Rect(*rect) for rect in segment["rects"]]
                annot = page.add_highlight_annot(rects)
                annot.set_colors(stroke=(0.20, 0.48, 0.95))
                annot.set_opacity(0.18)
                annot.set_info(
                    title="Paragraph Extraction Audit",
                    subject=f"{row['paragraph_id']} segment {segment_index}/{len(segments)}",
                    content=(
                        f"Paragraph ID: {row['paragraph_id']}\n"
                        f"Pages: {row['page_numbers']}\n"
                        f"Content type: {row['content_type']}\n"
                        f"Tokens: {row['token_count']}\n"
                        f"Cross-extractor similarity: {row['cross_extractor_similarity']:.3f}\n"
                        f"Merge reasons: {row['merge_reasons'] or 'None'}\n\n"
                        f"{row['text']}"
                    )[:7000],
                )
                annot.update()
                annotations_added += 1

        if include_excluded_overlay:
            excluded = line_audit[line_audit["role"].eq("excluded")]
            for _, row in excluded.iterrows():
                page = doc[int(row["page_number"]) - 1]
                rect = fitz.Rect(row["x0"], row["y0"], row["x1"], row["y1"])
                annot = page.add_rect_annot(rect)
                annot.set_colors(stroke=(0.60, 0.60, 0.60))
                annot.set_opacity(0.35)
                annot.set_border(width=0.3)
                annot.set_info(
                    title="Excluded Extraction Line",
                    subject=row["exclusion_reason"],
                    content=row["text"][:1500],
                )
                annot.update()
                annotations_added += 1

        doc.save(output_path, garbage=4, deflate=True)

    return {
        "source_pdf": str(source_path),
        "audit_pdf": str(output_path),
        "existing_annotations_removed": existing_removed,
        "audit_annotations_added": annotations_added,
    }


In [16]:
EXTRACTION_CONFIG = ExtractionConfig(
    engine=EXTRACTION_ENGINE,
    min_tokens=MIN_PARAGRAPH_TOKENS,
    max_tokens=MAX_PARAGRAPH_TOKENS,
    cross_extractor_min_median_similarity=CROSS_EXTRACTOR_MEDIAN_SIMILARITY,
    cross_extractor_min_paragraph_similarity=CROSS_EXTRACTOR_MIN_SIMILARITY,
    exclude_back_matter=EXCLUDE_BACK_MATTER,
    include_abstract=INCLUDE_ABSTRACT,
    include_block_quotes=INCLUDE_BLOCK_QUOTES,
    exclude_epigraphs=EXCLUDE_EPIGRAPHS,
)

pdf_paths = sorted(INPUT_DIR.glob(PDF_PATTERN))
if not pdf_paths:
    raise FileNotFoundError(
        f"No PDFs matching {PDF_PATTERN!r} were found in {INPUT_DIR}. Upload at least one source PDF."
    )

paragraph_frames = []
line_audit_frames = []
quality_check_frames = []
alignment_frames = []
page_agreement_frames = []
extraction_manifest_rows = []
extraction_audit_rows = []

for pdf_path in pdf_paths:
    result = extract_document(pdf_path, EXTRACTION_CONFIG)
    paragraphs = result["paragraphs"].copy()
    with fitz.open(pdf_path) as _source_doc:
        _metadata_title = clean_text((_source_doc.metadata or {}).get("title", ""))
    paragraphs["source_file"] = str(pdf_path)
    paragraphs["source_filename"] = pdf_path.name
    paragraphs["document_title"] = _metadata_title or pdf_path.stem
    paragraph_frames.append(paragraphs)

    line_audit = result["line_audit"].copy()
    line_audit["source_file"] = str(pdf_path)
    line_audit["source_filename"] = pdf_path.name
    line_audit_frames.append(line_audit)

    checks = result["quality_checks"].copy()
    checks.insert(0, "source_filename", pdf_path.name)
    checks.insert(1, "selected_engine", result["selected_engine"])
    quality_check_frames.append(checks)

    alignment = result["alignment"].copy()
    alignment.insert(0, "source_filename", pdf_path.name)
    alignment_frames.append(alignment)

    page_agreement = result["page_agreement"].copy()
    page_agreement.insert(0, "source_filename", pdf_path.name)
    page_agreement_frames.append(page_agreement)

    audit_path = EXTRACTION_AUDIT_DIR / f"{pdf_path.stem}_paragraph_extraction_audit.pdf"
    audit_rec = create_extraction_audit_pdf(
        pdf_path, paragraphs, line_audit, audit_path, include_excluded_overlay=False
    )
    extraction_audit_rows.append(audit_rec)

    extraction_manifest_rows.append({
        "source_filename": pdf_path.name,
        "selected_engine": result["selected_engine"],
        "alternate_engine": result["alternate_engine"],
        "body_font_size": result["body_font_size"],
        "logical_paragraphs": len(paragraphs),
        "cross_page_paragraphs": int(paragraphs["spans_multiple_pages"].sum()),
        "cross_column_paragraphs": int(paragraphs["spans_multiple_columns"].sum()),
        "manual_review_rows": int(paragraphs["manual_review_required"].sum()),
        "all_checks_passed": result["all_checks_passed"],
        "audit_pdf": str(audit_path),
    })

paragraphs_df = pd.concat(paragraph_frames, ignore_index=True)
line_audit_df = pd.concat(line_audit_frames, ignore_index=True)
extraction_quality_df = pd.concat(quality_check_frames, ignore_index=True)
extractor_alignment_df = pd.concat(alignment_frames, ignore_index=True)
page_extractor_agreement_df = pd.concat(page_agreement_frames, ignore_index=True)
extraction_manifest_df = pd.DataFrame(extraction_manifest_rows)
extraction_audit_df = pd.DataFrame(extraction_audit_rows)

if paragraphs_df["paragraph_id"].duplicated().any():
    raise RuntimeError("Duplicate paragraph IDs were generated.")
if paragraphs_df["manual_review_required"].any():
    display(paragraphs_df[paragraphs_df["manual_review_required"]])

all_extraction_checks_passed = bool(extraction_quality_df["passed"].all())
if FAIL_ON_EXTRACTION_CHECKS and not all_extraction_checks_passed:
    display(extraction_quality_df[~extraction_quality_df["passed"]])
    raise RuntimeError("Extraction quality checks failed. Do not run the model coders until they are resolved.")

paragraphs_df["duplicate_occurrence_count"] = paragraphs_df.groupby("text_hash")["text_hash"].transform("size")
unique_paragraphs_df = paragraphs_df.drop_duplicates("text_hash", keep="first").reset_index(drop=True)
if MAX_UNIQUE_PARAGRAPHS is not None:
    unique_paragraphs_df = unique_paragraphs_df.head(MAX_UNIQUE_PARAGRAPHS).copy()

paragraphs_df.to_csv(OUTPUT_DIR / "verified_logical_paragraphs.csv", index=False)
line_audit_df.to_csv(OUTPUT_DIR / "line_extraction_audit.csv", index=False)
extraction_quality_df.to_csv(OUTPUT_DIR / "extraction_quality_checks.csv", index=False)
extractor_alignment_df.to_csv(OUTPUT_DIR / "cross_extractor_paragraph_alignment.csv", index=False)
page_extractor_agreement_df.to_csv(OUTPUT_DIR / "cross_extractor_page_agreement.csv", index=False)
extraction_manifest_df.to_csv(OUTPUT_DIR / "extraction_manifest.csv", index=False)

EXTRACTION_REVIEW_WORKBOOK = OUTPUT_DIR / "paragraph_extraction_review.xlsx"
with pd.ExcelWriter(EXTRACTION_REVIEW_WORKBOOK, engine="xlsxwriter") as writer:
    extraction_manifest_df.to_excel(writer, sheet_name="Manifest", index=False)
    extraction_quality_df.to_excel(writer, sheet_name="Quality Checks", index=False)
    paragraphs_df.to_excel(writer, sheet_name="Logical Paragraphs", index=False)
    extractor_alignment_df.to_excel(writer, sheet_name="Extractor Alignment", index=False)
    page_extractor_agreement_df.to_excel(writer, sheet_name="Page Agreement", index=False)
    line_audit_df.to_excel(writer, sheet_name="Line Audit", index=False)
    extraction_audit_df.to_excel(writer, sheet_name="Audit PDFs", index=False)
    workbook = writer.book
    header = workbook.add_format({"bold": True, "font_color": "white", "bg_color": "#1F4E78", "border": 1})
    pass_format = workbook.add_format({"bg_color": "#C6EFCE", "font_color": "#006100"})
    fail_format = workbook.add_format({"bg_color": "#FFC7CE", "font_color": "#9C0006"})
    for sheet_name, frame in [
        ("Manifest", extraction_manifest_df), ("Quality Checks", extraction_quality_df),
        ("Logical Paragraphs", paragraphs_df), ("Extractor Alignment", extractor_alignment_df),
        ("Page Agreement", page_extractor_agreement_df), ("Line Audit", line_audit_df),
        ("Audit PDFs", extraction_audit_df),
    ]:
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, max(len(frame), 1), max(len(frame.columns)-1, 0))
        for col_idx, column in enumerate(frame.columns):
            ws.write(0, col_idx, column, header)
            width = min(max(len(str(column)) + 2, 12), 42)
            if column in {"text", "primary_text", "secondary_text", "segments_json"}:
                width = 45
            ws.set_column(col_idx, col_idx, width)
    if not extraction_quality_df.empty:
        passed_col = extraction_quality_df.columns.get_loc("passed")
        ws = writer.sheets["Quality Checks"]
        ws.conditional_format(1, passed_col, len(extraction_quality_df), passed_col,
                              {"type": "cell", "criteria": "==", "value": True, "format": pass_format})
        ws.conditional_format(1, passed_col, len(extraction_quality_df), passed_col,
                              {"type": "cell", "criteria": "==", "value": False, "format": fail_format})

print("Documents:", len(pdf_paths))
print("Verified logical paragraph occurrences:", len(paragraphs_df))
print("Unique paragraph texts to label:", len(unique_paragraphs_df))
print("Cross-page paragraphs:", int(paragraphs_df["spans_multiple_pages"].sum()))
print("Cross-column paragraphs:", int(paragraphs_df["spans_multiple_columns"].sum()))
print("All automated extraction checks passed:", all_extraction_checks_passed)
print("Extraction review workbook:", EXTRACTION_REVIEW_WORKBOOK)
display(extraction_manifest_df)
display(paragraphs_df[["paragraph_id", "start_page", "end_page", "content_type", "token_count", "merge_reasons", "text"]].head(12))

Documents: 1
Verified logical paragraph occurrences: 32
Unique paragraph texts to label: 32
Cross-page paragraphs: 5
Cross-column paragraphs: 10
All automated extraction checks passed: True
Extraction review workbook: /content/unlearning_pipeline/outputs/paragraph_extraction_review.xlsx


,source_filename,selected_engine,alternate_engine,body_font_size,logical_paragraphs,cross_page_paragraphs,cross_column_paragraphs,manual_review_rows,all_checks_passed,audit_pdf
0,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,pymupdf,pdfplumber,10.0,32,5,10,0,True,/content/unlearning_pipeline/outputs/extraction_audits/Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management ...


,paragraph_id,start_page,end_page,content_type,token_count,merge_reasons,text
0,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0001,1,1,abstract,100,,What would another Hurricane Katrina bring in 2020? Is government at all levels ready for such an event? This essay argues that learning to manage actions supporting disaster r...
1,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0002,1,1,body,136,cross_column_continuation,"Scores of reports, newspaper accounts, documentaries, and academic articles have been produced about the catastrophic U.S. hurricane of 2005, Hurricane Katrina. Many of these a..."
2,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0003,1,1,body,248,,"What would another Hurricane Katrina bring in 2020? Is government at all levels ready for such an event? Or is the worst yet to come? (Kettl 2006). In the following sections, w..."
3,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0004,1,2,body,212,cross_page_continuation,"Because much of the blame for the overall ineffective response to Katrina is placed on the Federal Emergency Management Agency (FEMA), as the nation’s coordinating arm for natu..."
4,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0005,2,2,body,188,,"It is important to note at the outset, particularly within the context of the failures of Hurricane Katrina, that FEMA, like many other large agencies such as NASA or the U.S. ..."
5,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0006,2,2,body,129,cross_column_continuation,"The emergency management field was a patchwork of relationships and requirements, and as broad as the type of risks that society faced. In addition to dealing with natural and ..."
6,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0007,2,2,body,84,,"Within two years after 9/11, FEMA had been absorbed into the newly cobbled together Department of Homeland Security along with more than 20 other organizational entities, many ..."
7,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0008,2,2,body,160,,"At the time of Hurricane Katrina, on August 29, 2005, emergency management and program content and fiscal support varied wildly by region and locality; state and local programs..."
8,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0009,2,2,body,114,,"The unevenness in the maturity and capacity of local emergency administrative infrastructures not only resulted in legitimate regional diff erences, but also translated into an..."
9,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0010,2,3,body,103,cross_page_continuation,"Furthermore, FEMA’s Federal Response Plan, an operational plan that laid out the individual roles and responsibilities of federal responders, had been replaced in the post-9/11..."


### Mandatory extraction approval

Open every PDF in `outputs/extraction_audits/`. Blue highlighting should cover every complete analytical paragraph exactly once, including continuations across columns and pages. Headings, pull quotes, biographies, repeated margins, epigraphs, and references should remain unhighlighted.

Only after visual review, set:
```python
EXTRACTION_APPROVED = True
```
The provider-run cell will otherwise stop before any paid request.

In [17]:
print("EXTRACTION_APPROVED =", EXTRACTION_APPROVED)
print("All automated checks passed =", all_extraction_checks_passed)

EXTRACTION_APPROVED = True
All automated checks passed = True


## 7. Select balanced few-shot examples and enforce leakage safeguards

In [18]:
def choose_balanced_examples(pool: pd.DataFrame, n_examples: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    chosen_indices = []
    labels = [False, True]

    per_label = n_examples // 2
    for label in labels:
        idx = pool.index[pool["example_unlearning"].eq(label)].to_numpy()
        take = min(per_label, len(idx))
        if take:
            chosen_indices.extend(rng.choice(idx, size=take, replace=False).tolist())

    remaining = n_examples - len(chosen_indices)
    if remaining > 0:
        available = pool.index[~pool.index.isin(chosen_indices)].to_numpy()
        if len(available):
            chosen_indices.extend(rng.choice(available, size=min(remaining, len(available)), replace=False).tolist())

    return pool.loc[chosen_indices].copy().reset_index(drop=True)


def document_key(value: str) -> str:
    key = normalize_for_match(value)
    key = re.sub(r"\bpdf\b", " ", key)
    return re.sub(r"\s+", " ", key).strip()


def audit_and_filter_examples(candidates: pd.DataFrame, input_paragraphs: pd.DataFrame):
    input_norms = input_paragraphs["normalized_text"].tolist()
    input_hashes = set(input_paragraphs["text_hash"])
    input_document_keys = set()
    for value in input_paragraphs["source_filename"].tolist() + input_paragraphs["document_title"].tolist():
        key = document_key(value)
        if key:
            input_document_keys.add(key)

    audit_rows = []
    kept_rows = []

    for _, row in candidates.iterrows():
        norm = normalize_for_match(row["Text Content"])
        text_hash = sha256_text(norm)
        exact_match = text_hash in input_hashes

        max_similarity = 0.0
        if input_norms:
            max_similarity = max(rapid_ratio(norm, target) / 100.0 for target in input_norms)

        example_doc_key = document_key(row.get("Document", ""))
        same_document = bool(example_doc_key and example_doc_key in input_document_keys)

        reasons = []
        if exact_match:
            reasons.append("exact_text_overlap")
        if max_similarity >= LEAKAGE_SIMILARITY_THRESHOLD:
            reasons.append("near_text_overlap")
        if EXCLUDE_SAME_DOCUMENT_EXAMPLES and same_document:
            reasons.append("same_document")

        keep = not reasons
        audit_rows.append({
            "example_number": row.get("Number", ""),
            "example_unlearning": row["example_unlearning"],
            "example_document": row.get("Document", ""),
            "exact_match": exact_match,
            "max_similarity": max_similarity,
            "same_document": same_document,
            "kept": keep,
            "exclusion_reason": "; ".join(reasons),
            "text_preview": row["Text Content"][:240],
        })
        if keep:
            kept_rows.append(row.to_dict())

    return pd.DataFrame(kept_rows), pd.DataFrame(audit_rows)


# Audit the full pool first, then select a balanced set from safe examples.
safe_example_pool_df, example_leakage_audit_df = audit_and_filter_examples(
    examples_pool_df,
    unique_paragraphs_df,
)
prompt_examples_df = choose_balanced_examples(
    safe_example_pool_df,
    N_FEWSHOT_EXAMPLES,
    FEWSHOT_SEED,
)

example_leakage_audit_df.to_csv(OUTPUT_DIR / "example_leakage_audit.csv", index=False)
prompt_examples_df.to_csv(OUTPUT_DIR / "fewshot_examples_used.csv", index=False)

required_by_strategy = any(cfg["strategy"] == "definitions_examples_no_metadata" for cfg in MODEL_CONFIGS)
if required_by_strategy and len(prompt_examples_df) < MIN_SAFE_FEWSHOT_EXAMPLES:
    raise RuntimeError(
        f"Only {len(prompt_examples_df)} safe few-shot examples remain after leakage checks. "
        f"At least {MIN_SAFE_FEWSHOT_EXAMPLES} are required. Add approved examples from other documents."
    )

print("Candidate examples:", len(examples_pool_df))
print("Safe examples after document/text leakage checks:", len(safe_example_pool_df))
print("Examples used in OpenAI/Anthropic prompts:", len(prompt_examples_df))
print("Selected label counts:", prompt_examples_df["example_unlearning"].value_counts().to_dict())
display(example_leakage_audit_df[~example_leakage_audit_df["kept"]].head(20))

Candidate examples: 49
Safe examples after document/text leakage checks: 49
Examples used in OpenAI/Anthropic prompts: 6
Selected label counts: {False: 3, True: 3}


,example_number,example_unlearning,example_document,exact_match,max_similarity,same_document,kept,exclusion_reason,text_preview


## 8. Shared structured output and provider-specific prompts

In [19]:
TARGET_TYPES = [
    "Leadership",
    "laws_plans_policies",
    "capabilities",
    "funds_resources",
    "misc_organizational",
    "none",
]

TARGET_DISPLAY = {
    "Leadership": "Leadership",
    "laws_plans_policies": "Laws, plans and policies",
    "capabilities": "Capabilities",
    "funds_resources": "Funds and resources",
    "misc_organizational": "Miscellaneous organizational",
    "none": "None",
    "needs_review": "Needs review",
}


class AnnotationOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    unlearning_present: bool
    target_type: Literal[
        "Leadership",
        "laws_plans_policies",
        "capabilities",
        "funds_resources",
        "misc_organizational",
        "none",
    ]
    agency: Optional[str] = None
    confidence: float = Field(ge=0.0, le=1.0)
    rationale: str


ANNOTATION_JSON_SCHEMA = AnnotationOutput.model_json_schema()
BASE_SYSTEM_PROMPT = "Return only the requested structured output. Do not include markdown."

COMMON_TASK_BLOCK = """You are coding federal disaster-policy text for organizational unlearning.

Task:
1. Decide whether the target paragraph contains unlearning.
2. If yes, assign exactly one target type.
3. Identify the agency or sub-unit only when stated or clearly identifiable in the paragraph.
4. Give one concise rationale grounded in the paragraph.

Unlearning target types:
- Leadership
- laws_plans_policies
- capabilities
- funds_resources
- misc_organizational
- none

When unlearning_present is false, target_type must be none.""".strip()


def example_codes(row: pd.Series) -> str:
    codes = clean_text(row.get("Codes", ""))
    if codes:
        return codes

    parts = [f"Unlearning: {'Yes' if row['example_unlearning'] else 'No'}"]
    target = clean_text(row.get("Target", ""))
    agency = clean_text(row.get("Government Agency", ""))
    if target:
        parts.append(f"Target: {target}")
    if agency:
        parts.append(f"Government Agency: {agency}")
    return " | ".join(parts)


def serialize_examples_no_metadata(df: pd.DataFrame) -> str:
    blocks = []
    for i, (_, row) in enumerate(df.iterrows(), start=1):
        blocks.append("\n".join([
            f"Example {i}",
            "Text Content:",
            row["Text Content"],
            "Human labels:",
            f"Unlearning: {'Yes' if row['example_unlearning'] else 'No'}",
            f"Codes: {example_codes(row)}",
        ]))
    return "\n\n---\n\n".join(blocks)


EXAMPLES_TEXT_NO_METADATA = serialize_examples_no_metadata(prompt_examples_df)


def build_user_prompt(strategy: str, row: pd.Series) -> str:
    paragraph_id = row.get("paragraph_id", row.get("text_hash", "paragraph"))
    paragraph = row["text"]

    if strategy == "direct_no_context":
        return f"""{COMMON_TASK_BLOCK}

Paragraph ID: {paragraph_id}
Paragraph:
{paragraph}""".strip()

    if strategy == "definitions_examples_no_metadata":
        return f"""{COMMON_TASK_BLOCK}

Codebook:
{CODEBOOK_TEXT}

Labeled examples:
{EXAMPLES_TEXT_NO_METADATA}

Paragraph ID: {paragraph_id}
Paragraph:
{paragraph}""".strip()

    raise ValueError(f"Unsupported strategy: {strategy}")


def build_messages(cfg: dict, row: pd.Series) -> list[dict]:
    return [
        {"role": "system", "content": BASE_SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(cfg["strategy"], row)},
    ]


def prompt_hash(cfg: dict, row: pd.Series) -> str:
    payload = {
        "prompt_version": PROMPT_VERSION,
        "provider": cfg["provider"],
        "model": cfg["model"],
        "strategy": cfg["strategy"],
        "messages": build_messages(cfg, row),
        "schema": ANNOTATION_JSON_SCHEMA,
    }
    return sha256_text(json.dumps(payload, ensure_ascii=False, sort_keys=True))


preview = unique_paragraphs_df.iloc[0]
for cfg in MODEL_CONFIGS:
    prompt = build_messages(cfg, preview)[1]["content"]
    print("\n", cfg["provider"], cfg["strategy"], "characters:", len(prompt))
    print(prompt[:700], "...")


 openai definitions_examples_no_metadata characters: 18562
You are coding federal disaster-policy text for organizational unlearning.

Task:
1. Decide whether the target paragraph contains unlearning.
2. If yes, assign exactly one target type.
3. Identify the agency or sub-unit only when stated or clearly identifiable in the paragraph.
4. Give one concise rationale grounded in the paragraph.

Unlearning target types:
- Leadership
- laws_plans_policies
- capabilities
- funds_resources
- misc_organizational
- none

When unlearning_present is false, target_type must be none.

Codebook:
Code: Step 1 -- Binary classification based on unlearning definition
Definition: 
Detection logic: 
Examples: 
Positive clarification: 
Negative clarification: 

Code: U ...

 anthropic definitions_examples_no_metadata characters: 18562
You are coding federal disaster-policy text for organizational unlearning.

Task:
1. Decide whether the target paragraph contains unlearning.
2. If yes, assign exactly one 

## 9. Provider API calls, retry handling, and resumable JSONL logs

In [20]:
class NonRetryableRequestError(RuntimeError):
    pass


def retry_call(fn, *args, max_attempts=MAX_RETRY_ATTEMPTS, **kwargs):
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(*args, **kwargs)
        except NonRetryableRequestError:
            raise
        except Exception as exc:
            last_error = exc
            message = str(exc).lower()
            nonretryable_markers = [
                "invalid_request_error", "error code: 400", "400 invalid_argument",
                "unknown name", "deprecated for this model", "cannot find field",
            ]
            if any(marker in message for marker in nonretryable_markers):
                raise NonRetryableRequestError(str(exc)) from exc
            if attempt == max_attempts:
                raise
            wait = min(60.0, 4.0 * (2 ** (attempt - 1)))
            print(f"Transient error ({attempt}/{max_attempts}): {exc}\nRetrying in {wait:.1f}s...")
            time.sleep(wait)
    raise last_error


def usage_result(raw_response, input_tokens=0, output_tokens=0, total_tokens=0, reasoning_tokens=0, latency=0.0):
    return {
        "raw_response": raw_response,
        "input_tokens": int(input_tokens or 0),
        "output_tokens": int(output_tokens or 0),
        "total_tokens": int(total_tokens or (input_tokens or 0) + (output_tokens or 0)),
        "reasoning_tokens": int(reasoning_tokens or 0),
        "latency_seconds": float(latency),
    }


def call_openai(messages: list[dict], cfg: dict) -> dict:
    from openai import OpenAI

    client = OpenAI(api_key=os.environ[cfg["api_key_env"]])
    system_text = next(m["content"] for m in messages if m["role"] == "system")
    response_input = [m for m in messages if m["role"] != "system"]

    start = time.time()
    response = client.responses.parse(
        model=cfg["model"],
        instructions=system_text,
        input=response_input,
        reasoning={"effort": cfg.get("reasoning_effort", "low")},
        max_output_tokens=MAX_OUTPUT_TOKENS,
        text_format=AnnotationOutput,
        store=False,
    )
    latency = time.time() - start

    parsed = response.output_parsed
    if parsed is None:
        raise RuntimeError(f"OpenAI returned no parsed output: {response.output_text!r}")

    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", 0) or 0
    output_tokens = getattr(usage, "output_tokens", 0) or 0
    total_tokens = getattr(usage, "total_tokens", 0) or input_tokens + output_tokens
    details = getattr(usage, "output_tokens_details", None)
    reasoning_tokens = getattr(details, "reasoning_tokens", 0) or 0

    return usage_result(
        json.dumps(parsed.model_dump(), ensure_ascii=False),
        input_tokens, output_tokens, total_tokens, reasoning_tokens, latency,
    )


def call_anthropic(messages: list[dict], cfg: dict) -> dict:
    from anthropic import Anthropic

    client = Anthropic(api_key=os.environ[cfg["api_key_env"]])
    system_text = "\n".join(m["content"] for m in messages if m["role"] == "system")
    anthropic_messages = [m for m in messages if m["role"] in {"user", "assistant"}]

    request = {
        "model": cfg["model"],
        "max_tokens": MAX_OUTPUT_TOKENS,
        "system": system_text,
        "messages": anthropic_messages,
        "output_format": AnnotationOutput,
    }
    if cfg.get("effort") is not None:
        request["output_config"] = {"effort": cfg["effort"]}

    start = time.time()
    response = client.messages.parse(**request)
    latency = time.time() - start

    parsed = response.parsed_output
    if parsed is None:
        raise RuntimeError("Anthropic returned no parsed output.")

    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", 0) or 0
    output_tokens = getattr(usage, "output_tokens", 0) or 0

    return usage_result(
        json.dumps(parsed.model_dump(), ensure_ascii=False),
        input_tokens, output_tokens, input_tokens + output_tokens, 0, latency,
    )


def call_gemini(messages: list[dict], cfg: dict) -> dict:
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=os.environ[cfg["api_key_env"]])
    system_text = next(m["content"] for m in messages if m["role"] == "system")
    user_text = "\n\n".join(m["content"] for m in messages if m["role"] == "user")

    start = time.time()
    response = client.models.generate_content(
        model=cfg["model"],
        contents=user_text,
        config=types.GenerateContentConfig(
            system_instruction=system_text,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            response_mime_type="application/json",
            response_json_schema=ANNOTATION_JSON_SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level=cfg.get("thinking_level", "minimal")
            ),
        ),
    )
    latency = time.time() - start

    raw = response.text or ""
    try:
        parsed = AnnotationOutput.model_validate_json(raw)
    except ValidationError as exc:
        raise RuntimeError(f"Gemini structured output failed validation: {exc}; raw={raw!r}") from exc

    usage = getattr(response, "usage_metadata", None)
    input_tokens = getattr(usage, "prompt_token_count", 0) or 0
    output_tokens = getattr(usage, "candidates_token_count", 0) or 0
    total_tokens = getattr(usage, "total_token_count", 0) or input_tokens + output_tokens
    thinking_tokens = getattr(usage, "thoughts_token_count", 0) or 0

    return usage_result(
        json.dumps(parsed.model_dump(), ensure_ascii=False),
        input_tokens, output_tokens, total_tokens, thinking_tokens, latency,
    )


def mock_call(messages: list[dict], cfg: dict) -> dict:
    """Deterministic smoke-test output. Never use as research data."""
    text = messages[-1]["content"].rsplit("Paragraph:\n", 1)[-1].lower()
    positive_terms = ["replace", "abandon", "move away", "eliminate", "reorganize", "reform"]
    present = any(term in text for term in positive_terms)
    target = "laws_plans_policies" if present else "none"
    parsed = AnnotationOutput(
        unlearning_present=present,
        target_type=target,
        agency=None,
        confidence=0.75,
        rationale="Mock-mode deterministic classification for pipeline testing only.",
    )
    return usage_result(json.dumps(parsed.model_dump()), latency=0.001)


def call_provider(messages: list[dict], cfg: dict) -> dict:
    if MOCK_MODE:
        return mock_call(messages, cfg)
    if cfg["provider"] == "openai":
        return retry_call(call_openai, messages, cfg)
    if cfg["provider"] == "anthropic":
        return retry_call(call_anthropic, messages, cfg)
    if cfg["provider"] == "gemini":
        return retry_call(call_gemini, messages, cfg)
    raise ValueError(f"Unsupported provider: {cfg['provider']}")


def provider_jsonl_path(cfg: dict) -> Path:
    return RAW_DIR / f"{cfg['provider']}__{safe_slug(cfg['model'])}__{PROMPT_VERSION}.jsonl"


def load_success_records(path: Path) -> dict[str, dict]:
    records = {}
    if not path.exists() or not RESUME_FROM_JSONL:
        return records
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            rec = json.loads(line)
            if rec.get("status") == "ok" and rec.get("run_key"):
                records[rec["run_key"]] = rec
    return records


def append_jsonl(path: Path, record: dict):
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def semantic_validate(parsed: AnnotationOutput) -> list[str]:
    warnings_list = []
    if not parsed.unlearning_present and parsed.target_type != "none":
        warnings_list.append("negative_prediction_with_non_none_target")
    if parsed.unlearning_present and parsed.target_type == "none":
        warnings_list.append("positive_prediction_with_none_target")
    return warnings_list


def run_provider(cfg: dict, rows: pd.DataFrame) -> pd.DataFrame:
    path = provider_jsonl_path(cfg)
    prior = load_success_records(path)
    results = []

    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"{cfg['provider']} labeling"):
        p_hash = prompt_hash(cfg, row)
        run_key = sha256_text("|".join([
            PROMPT_VERSION, cfg["provider"], cfg["model"], cfg["strategy"],
            row["text_hash"], p_hash,
        ]))

        if run_key in prior:
            results.append(prior[run_key])
            continue

        base = {
            "run_key": run_key,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "prompt_version": PROMPT_VERSION,
            "prompt_sha256": p_hash,
            "provider": cfg["provider"],
            "model": cfg["model"],
            "strategy": cfg["strategy"],
            "text_hash": row["text_hash"],
            "representative_paragraph_id": row["paragraph_id"],
            "representative_document": row["source_filename"],
            "representative_page": int(row["start_page"]),
            "text": row["text"],
        }

        if not RUN_API_CALLS and not MOCK_MODE:
            record = {
                **base,
                "status": "dry_run",
                "pred_unlearning": None,
                "pred_target_type": None,
                "pred_agency": None,
                "confidence": None,
                "rationale": None,
                "semantic_warning": None,
                "raw_response": None,
                "input_tokens": 0,
                "output_tokens": 0,
                "reasoning_tokens": 0,
                "total_tokens": 0,
                "latency_seconds": 0.0,
                "error": None,
            }
        else:
            try:
                api_result = call_provider(build_messages(cfg, row), cfg)
                parsed = AnnotationOutput.model_validate_json(api_result["raw_response"])
                semantic_warnings = semantic_validate(parsed)
                target = parsed.target_type if parsed.unlearning_present else "none"

                record = {
                    **base,
                    "status": "ok",
                    "pred_unlearning": bool(parsed.unlearning_present),
                    "pred_target_type": target,
                    "pred_agency": clean_text(parsed.agency),
                    "confidence": float(parsed.confidence),
                    "rationale": clean_text(parsed.rationale),
                    "semantic_warning": "; ".join(semantic_warnings),
                    **api_result,
                    "error": None,
                }
            except Exception as exc:
                record = {
                    **base,
                    "status": "error",
                    "pred_unlearning": None,
                    "pred_target_type": None,
                    "pred_agency": None,
                    "confidence": None,
                    "rationale": None,
                    "semantic_warning": None,
                    "raw_response": None,
                    "input_tokens": 0,
                    "output_tokens": 0,
                    "reasoning_tokens": 0,
                    "total_tokens": 0,
                    "latency_seconds": 0.0,
                    "error": repr(exc),
                }

        append_jsonl(path, record)
        results.append(record)
        time.sleep(REQUEST_SLEEP_SECONDS)

    result_df = pd.DataFrame(results)
    result_df.to_csv(OUTPUT_DIR / f"{cfg['provider']}_predictions_snapshot.csv", index=False)
    return result_df

print("Provider functions loaded.")

Provider functions loaded.


## 10. Run the three providers

In [22]:
if not all_extraction_checks_passed:
    raise RuntimeError("Automated extraction checks did not pass. Model calls are blocked.")
if RUN_API_CALLS and not EXTRACTION_APPROVED:
    raise RuntimeError(
        "Model calls are blocked until the extraction-audit PDFs have been reviewed. "
        "Set EXTRACTION_APPROVED = True after visual approval."
    )

provider_result_frames = []
for cfg in MODEL_CONFIGS:
    frame = run_provider(cfg, unique_paragraphs_df)
    provider_result_frames.append(frame)

model_predictions_unique_df = pd.concat(provider_result_frames, ignore_index=True)
model_predictions_unique_df.to_csv(OUTPUT_DIR / "model_predictions_unique_texts.csv", index=False)

status_summary = model_predictions_unique_df.groupby(["provider", "status"]).size().unstack(fill_value=0)
display(status_summary)

if not MOCK_MODE and RUN_API_CALLS:
    errors = model_predictions_unique_df[model_predictions_unique_df["status"].ne("ok")]
    if not errors.empty:
        display(errors[["provider", "representative_paragraph_id", "error"]].head(20))
        raise RuntimeError(
            f"{len(errors)} provider calls did not complete successfully. Fix them and rerun; completed calls will resume."
        )

if not RUN_API_CALLS and not MOCK_MODE:
    raise RuntimeError(
        "Dry run completed, but no labels were generated. Set UNLEARNING_RUN_API_CALLS=1 or RUN_API_CALLS=True and rerun from the API-key cell."
    )

openai labeling:   0%|          | 0/32 [00:00<?, ?it/s]

anthropic labeling:   0%|          | 0/32 [00:00<?, ?it/s]

gemini labeling:   0%|          | 0/32 [00:00<?, ?it/s]

status,ok
provider,
anthropic,32
gemini,32
openai,32


## 11. Expand cached predictions back to every logical paragraph occurrence

In [23]:
valid_unique_predictions_df = model_predictions_unique_df[
    model_predictions_unique_df["status"].eq("ok")
].copy()

expected_providers = {cfg["provider"] for cfg in MODEL_CONFIGS}
observed_providers = set(valid_unique_predictions_df["provider"].unique())
if observed_providers != expected_providers:
    raise RuntimeError(f"Expected providers {expected_providers}, observed {observed_providers}.")

counts_per_hash = valid_unique_predictions_df.groupby("text_hash")["provider"].nunique()
if not counts_per_hash.eq(len(expected_providers)).all():
    bad_hashes = counts_per_hash[counts_per_hash.ne(len(expected_providers))]
    raise RuntimeError(f"Some paragraph texts are missing provider labels: {bad_hashes.head().to_dict()}")

model_predictions_long_df = paragraphs_df.merge(
    valid_unique_predictions_df.drop(columns=["text", "representative_paragraph_id", "representative_document", "representative_page"]),
    on="text_hash",
    how="left",
    validate="many_to_many",
)
model_predictions_long_df.to_csv(OUTPUT_DIR / "model_predictions_long.csv", index=False)

wide_parts = []
for provider in sorted(expected_providers):
    part = valid_unique_predictions_df[valid_unique_predictions_df["provider"].eq(provider)].copy()
    part = part[[
        "text_hash", "model", "strategy", "pred_unlearning", "pred_target_type",
        "pred_agency", "confidence", "rationale", "semantic_warning",
    ]]
    part = part.rename(columns={
        col: f"{provider}_{col}" for col in part.columns if col != "text_hash"
    })
    wide_parts.append(part)

model_predictions_wide_unique_df = wide_parts[0]
for part in wide_parts[1:]:
    model_predictions_wide_unique_df = model_predictions_wide_unique_df.merge(
        part, on="text_hash", how="inner", validate="one_to_one"
    )

model_predictions_wide_df = paragraphs_df.merge(
    model_predictions_wide_unique_df,
    on="text_hash",
    how="left",
    validate="many_to_one",
)
model_predictions_wide_df.to_csv(OUTPUT_DIR / "model_predictions_wide.csv", index=False)

print("Long prediction rows:", len(model_predictions_long_df))
print("Wide paragraph-occurrence rows:", len(model_predictions_wide_df))
display(model_predictions_wide_df.head())

Long prediction rows: 96
Wide paragraph-occurrence rows: 32


,source_file,source_filename,doc_id,extraction_engine,paragraph_id,paragraph_order,start_page,end_page,page_numbers,spans_multiple_pages,spans_multiple_columns,content_type,section_heading,token_count,line_count,text,normalized_text,text_hash,segments_json,merge_reasons,starts_lowercase,ends_terminal,dangling_nonterminal,geometry_valid,geometry_issues,cross_extractor_similarity,manual_review_required,document_title,duplicate_occurrence_count,anthropic_model,anthropic_strategy,anthropic_pred_unlearning,anthropic_pred_target_type,anthropic_pred_agency,anthropic_confidence,anthropic_rationale,anthropic_semantic_warning,gemini_model,gemini_strategy,gemini_pred_unlearning,gemini_pred_target_type,gemini_pred_agency,gemini_confidence,gemini_rationale,gemini_semantic_warning,openai_model,openai_strategy,openai_pred_unlearning,openai_pred_target_type,openai_pred_agency,openai_confidence,openai_rationale,openai_semantic_warning
0,/content/unlearning_pipeline/input_documents/Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020_,pymupdf,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0001,1,1,1,1,False,False,abstract,,100,17,What would another Hurricane Katrina bring in 2020? Is government at all levels ready for such an event? This essay argues that learning to manage actions supporting disaster r...,what would another hurricane katrina bring in 2020 is government at all levels ready for such an event this essay argues that learning to manage actions supporting disaster res...,995533a33280efd2a02b8fa89071cf45aefcf80b8503cf8bcb80188b6a2c3420,"[{""page_number"": 1, ""rects"": [[31.5, 223.0184783935547, 240.91000366210938, 233.0184783935547], [31.5, 235.0184783935547, 234.7360076904297, 245.0184783935547], [31.5, 247.0184...",,False,True,False,True,,1.0,False,What if Hurricane Katrina Hit in 2020? The Need for Strategic Management of Disasters,1,claude-sonnet-5,definitions_examples_no_metadata,False,none,,0.85,"The paragraph is an introductory/abstract framing discussion noting progress since Katrina and calling for 'greater strategic capacity,' but it does not name a specific prior p...",,gemini-3.1-flash-lite,direct_no_context,False,none,,0.9,"The paragraph discusses the need for future strategic learning and capacity development, but it does not describe the active abandonment or modification of existing organizatio...",,gpt-5.6-terra,definitions_examples_no_metadata,False,none,,0.96,"The paragraph advocates greater strategic capacity and contrasts strategic with reactive thinking, but it does not explicitly identify a prior Katrina-era practice as inadequat...",
1,/content/unlearning_pipeline/input_documents/Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020_,pymupdf,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0002,2,1,1,1,False,True,body,Strategic Management of Disasters Management,136,19,"Scores of reports, newspaper accounts, documentaries, and academic articles have been produced about the catastrophic U.S. hurricane of 2005, Hurricane Katrina. Many of these a...",scores of reports newspaper accounts documentaries and academic articles have been produced about the catastrophic u s hurricane of 2005 hurricane katrina many of these are emp...,e6dbb17fa4b8a09d1734720a69e3cb403870040260c3fd3524ecf1a2799ff46b,"[{""page_number"": 1, ""rects"": [[54.368499755859375, 559.0137329101562, 224.6284637451172, 569.

## 12. Inter-coder reliability

In [24]:
PROVIDER_ORDER = [cfg["provider"] for cfg in MODEL_CONFIGS]


def krippendorff_alpha_nominal(ratings: pd.DataFrame) -> float:
    """Nominal alpha for a units x coders matrix, allowing missing ratings."""
    array = ratings.to_numpy(dtype=object)
    categories = sorted({x for x in array.ravel() if pd.notna(x)}, key=str)
    if not categories:
        return np.nan

    observed_num = 0.0
    observed_den = 0.0
    total_counts = Counter()

    for row in array:
        values = [x for x in row if pd.notna(x)]
        n = len(values)
        if n < 2:
            continue
        counts = Counter(values)
        observed_num += sum(count * (n - count) for count in counts.values())
        observed_den += n * (n - 1)
        total_counts.update(values)

    if observed_den == 0:
        return np.nan
    observed_disagreement = observed_num / observed_den

    total_n = sum(total_counts.values())
    if total_n < 2:
        return np.nan
    expected_num = sum(count * (total_n - count) for count in total_counts.values())
    expected_den = total_n * (total_n - 1)
    expected_disagreement = expected_num / expected_den

    if expected_disagreement == 0:
        return 1.0 if observed_disagreement == 0 else np.nan
    return 1.0 - observed_disagreement / expected_disagreement


def fleiss_from_ratings(ratings: pd.DataFrame) -> float:
    categories = sorted(pd.unique(ratings.to_numpy().ravel()).tolist(), key=str)
    categories = [c for c in categories if pd.notna(c)]
    complete = ratings.dropna()
    if complete.empty or not categories:
        return np.nan

    count_matrix = np.array([
        [(row == category).sum() for category in categories]
        for row in complete.to_numpy(dtype=object)
    ], dtype=float)

    try:
        return float(fleiss_kappa(count_matrix, method="fleiss"))
    except Exception:
        return np.nan


def pairwise_reliability(ratings: pd.DataFrame, dimension: str, scope: str) -> pd.DataFrame:
    rows = []
    for coder_a, coder_b in combinations(ratings.columns, 2):
        pair = ratings[[coder_a, coder_b]].dropna()
        if pair.empty:
            continue
        agreement = float((pair[coder_a] == pair[coder_b]).mean())
        try:
            kappa = float(cohen_kappa_score(pair[coder_a], pair[coder_b]))
        except Exception:
            kappa = np.nan
        rows.append({
            "dimension": dimension,
            "scope": scope,
            "coder_a": coder_a,
            "coder_b": coder_b,
            "n_units": len(pair),
            "percent_agreement": agreement,
            "cohen_kappa": kappa,
        })
    return pd.DataFrame(rows)


def overall_reliability(ratings: pd.DataFrame, dimension: str, scope: str) -> dict:
    complete = ratings.dropna()
    return {
        "dimension": dimension,
        "scope": scope,
        "n_units_total": len(ratings),
        "n_units_complete": len(complete),
        "n_coders": len(ratings.columns),
        "unanimous_agreement": float(complete.nunique(axis=1).eq(1).mean()) if len(complete) else np.nan,
        "fleiss_kappa": fleiss_from_ratings(ratings),
        "krippendorff_alpha_nominal": krippendorff_alpha_nominal(ratings),
    }


def ratings_matrix(source: pd.DataFrame, value_column: str, index_column: str) -> pd.DataFrame:
    return source.pivot_table(
        index=index_column,
        columns="provider",
        values=value_column,
        aggfunc="first",
    ).reindex(columns=PROVIDER_ORDER)


reliability_pairwise_frames = []
reliability_overall_rows = []

for scope, source, index_col in [
    ("unique_text", valid_unique_predictions_df, "text_hash"),
    ("all_occurrences", model_predictions_long_df, "paragraph_id"),
]:
    binary = ratings_matrix(source, "pred_unlearning", index_col)
    target = ratings_matrix(source, "pred_target_type", index_col)

    reliability_pairwise_frames.append(pairwise_reliability(binary, "unlearning_binary", scope))
    reliability_pairwise_frames.append(pairwise_reliability(target, "target_including_none", scope))
    reliability_overall_rows.append(overall_reliability(binary, "unlearning_binary", scope))
    reliability_overall_rows.append(overall_reliability(target, "target_including_none", scope))

    any_positive = binary.eq(True).any(axis=1)
    target_any_positive = target.loc[any_positive]
    reliability_pairwise_frames.append(
        pairwise_reliability(target_any_positive, "target_on_any_model_positive", scope)
    )
    reliability_overall_rows.append(
        overall_reliability(target_any_positive, "target_on_any_model_positive", scope)
    )

reliability_pairwise_df = pd.concat(reliability_pairwise_frames, ignore_index=True)
reliability_overall_df = pd.DataFrame(reliability_overall_rows)

# Per-document binary reliability on paragraph occurrences.
per_document_rows = []
for document, subset in model_predictions_long_df.groupby("source_filename"):
    matrix = ratings_matrix(subset, "pred_unlearning", "paragraph_id")
    rec = overall_reliability(matrix, "unlearning_binary", "document_occurrences")
    rec["source_filename"] = document
    per_document_rows.append(rec)
reliability_by_document_df = pd.DataFrame(per_document_rows)

reliability_pairwise_df.to_csv(OUTPUT_DIR / "reliability_pairwise.csv", index=False)
reliability_overall_df.to_csv(OUTPUT_DIR / "reliability_overall.csv", index=False)
reliability_by_document_df.to_csv(OUTPUT_DIR / "reliability_by_document.csv", index=False)

display(reliability_overall_df)
display(reliability_pairwise_df)

,dimension,scope,n_units_total,n_units_complete,n_coders,unanimous_agreement,fleiss_kappa,krippendorff_alpha_nominal
0,unlearning_binary,unique_text,32,32,3,0.687500,0.484702,0.490070
1,target_including_none,unique_text,32,32,3,0.656250,0.438596,0.444444
2,target_on_any_model_positive,unique_text,14,14,3,0.214286,0.151515,0.171717
3,unlearning_binary,all_occurrences,32,32,3,0.687500,0.484702,0.490070
4,target_including_none,all_occurrences,32,32,3,0.656250,0.438596,0.444444
5,target_on_any_model_positive,all_occurrences,14,14,3,0.214286,0.151515,0.171717


,dimension,scope,coder_a,coder_b,n_units,percent_agreement,cohen_kappa
0,unlearning_binary,unique_text,openai,anthropic,32,0.843750,0.518072
1,unlearning_binary,unique_text,openai,gemini,32,0.750000,0.457627
2,unlearning_binary,unique_text,anthropic,gemini,32,0.781250,0.529412
3,target_including_none,unique_text,openai,anthropic,32,0.812500,0.457627
4,target_including_none,unique_text,openai,gemini,32,0.687500,0.382239
5,target_including_none,unique_text,anthropic,gemini,32,0.750000,0.512381
6,target_on_any_model_positive,unique_text,openai,anthropic,14,0.571429,0.300000
7,target_on_any_model_positive,unique_text,openai,gemini,14,0.285714,0.113924
8,target_on_any_model_positive,unique_text,anthropic,gemini,14,0.428571,0.238095
9,unlearning_binary,all_occurrences,openai,anthropic,32,0.843750,0.518072


## 13. Consensus labels and review priority

In [25]:
def consensus_for_text_hash(group: pd.DataFrame) -> dict:
    group = group.set_index("provider").reindex(PROVIDER_ORDER).reset_index()
    yes_votes = int(group["pred_unlearning"].eq(True).sum())
    no_votes = int(group["pred_unlearning"].eq(False).sum())
    consensus_yes = yes_votes >= 2

    if yes_votes == 3:
        agreement_bucket = "unanimous_yes"
    elif yes_votes == 2:
        agreement_bucket = "majority_yes"
    elif yes_votes == 1:
        agreement_bucket = "majority_no_one_positive"
    else:
        agreement_bucket = "unanimous_no"

    positive_group = group[group["pred_unlearning"].eq(True)]
    target_consensus = "none"
    target_vote_summary = ""
    target_agreement = True

    if consensus_yes:
        counts = Counter(positive_group["pred_target_type"].dropna().tolist())
        target_vote_summary = "; ".join(f"{k}:{v}" for k, v in counts.most_common())
        if counts:
            top_target, top_count = counts.most_common(1)[0]
            if top_count >= 2 or len(counts) == 1:
                target_consensus = top_target
            else:
                target_consensus = "needs_review"
                target_agreement = False
        else:
            target_consensus = "needs_review"
            target_agreement = False

    agreeing = group[group["pred_unlearning"].eq(consensus_yes)]
    consensus_confidence = float(agreeing["confidence"].mean()) if not agreeing.empty else np.nan

    agencies = sorted({
        clean_text(value)
        for value in positive_group["pred_agency"].tolist()
        if clean_text(value)
    })

    rationale_source = agreeing.sort_values("confidence", ascending=False).head(1)
    consensus_rationale = clean_text(rationale_source["rationale"].iloc[0]) if not rationale_source.empty else ""

    if agreement_bucket in {"majority_yes", "majority_no_one_positive"} or not target_agreement:
        review_priority = "High"
    elif agreement_bucket == "unanimous_yes":
        review_priority = "Medium"
    else:
        review_priority = "Low"

    result = {
        "text_hash": group["text_hash"].iloc[0],
        "yes_votes": yes_votes,
        "no_votes": no_votes,
        "consensus_unlearning": consensus_yes,
        "agreement_bucket": agreement_bucket,
        "consensus_target_type": target_consensus,
        "consensus_target_display": TARGET_DISPLAY.get(target_consensus, target_consensus),
        "target_vote_summary": target_vote_summary,
        "consensus_agencies": "; ".join(agencies),
        "consensus_confidence": consensus_confidence,
        "consensus_rationale": consensus_rationale,
        "review_priority": review_priority,
    }

    for _, row in group.iterrows():
        prefix = row["provider"]
        result[f"{prefix}_model"] = row["model"]
        result[f"{prefix}_strategy"] = row["strategy"]
        result[f"{prefix}_unlearning"] = row["pred_unlearning"]
        result[f"{prefix}_target_type"] = row["pred_target_type"]
        result[f"{prefix}_agency"] = row["pred_agency"]
        result[f"{prefix}_confidence"] = row["confidence"]
        result[f"{prefix}_rationale"] = row["rationale"]

    return result


consensus_unique_df = pd.DataFrame([
    consensus_for_text_hash(group)
    for _, group in valid_unique_predictions_df.groupby("text_hash", sort=False)
])

consensus_annotations_df = paragraphs_df.merge(
    consensus_unique_df,
    on="text_hash",
    how="left",
    validate="many_to_one",
)
consensus_annotations_df.to_csv(OUTPUT_DIR / "consensus_annotations.csv", index=False)

print(consensus_annotations_df["agreement_bucket"].value_counts())
display(consensus_annotations_df[[
    "paragraph_id", "source_filename", "start_page", "end_page", "agreement_bucket",
    "consensus_unlearning", "consensus_target_display", "review_priority", "text",
]].head(20))

agreement_bucket
unanimous_no                18
majority_no_one_positive     5
majority_yes                 5
unanimous_yes                4
Name: count, dtype: int64


,paragraph_id,source_filename,start_page,end_page,agreement_bucket,consensus_unlearning,consensus_target_display,review_priority,text
0,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0001,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,1,unanimous_no,False,None,Low,What would another Hurricane Katrina bring in 2020? Is government at all levels ready for such an event? This essay argues that learning to manage actions supporting disaster r...
1,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0002,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,1,majority_no_one_positive,False,None,High,"Scores of reports, newspaper accounts, documentaries, and academic articles have been produced about the catastrophic U.S. hurricane of 2005, Hurricane Katrina. Many of these a..."
2,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0003,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,1,unanimous_no,False,None,Low,"What would another Hurricane Katrina bring in 2020? Is government at all levels ready for such an event? Or is the worst yet to come? (Kettl 2006). In the following sections, w..."
3,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0004,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,2,unanimous_no,False,None,Low,"Because much of the blame for the overall ineffective response to Katrina is placed on the Federal Emergency Management Agency (FEMA), as the nation’s coordinating arm for natu..."
4,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0005,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,2,2,majority_no_one_positive,False,None,High,"It is important to note at the outset, particularly within the context of the failures of Hurricane Katrina, that FEMA, like many other large agencies such as NASA or the U.S. ..."
5,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0006,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,2,2,unanimous_no,False,None,Low,"The emergency management field was a patchwork of relationships and requirements, and as broad as the type of risks that society faced. In addition to dealing with natural and ..."
6,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0007,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,2,2,majority_no_one_positive,False,None,High,"Within two years after 9/11, FEMA had been absorbed into the newly cobbled together Department of Homeland Security along with more than 20 other organizational entities, many ..."
7,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0008,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,2,2,unanimous_no,False,None,Low,"At the time of Hurricane Katrina, on August 29, 2005, emergency management and program content and fiscal support varied wildly by region and locality; state and local programs..."
8,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hit_in_2020__para_0009,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,2,2,unanimous_no,False,None,Low,"The unevenness in the maturity and capacity of local emergency administrative infrastructures not only r

## 14. Export one review workbook

In [26]:
RESULTS_WORKBOOK_PATH = OUTPUT_DIR / "unlearning_annotation_results.xlsx"

run_manifest_df = pd.DataFrame([{
    "prompt_version": PROMPT_VERSION,
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "base_directory": str(BASE_DIR),
    "codebook_workbook": str(CODEBOOK_WORKBOOK_PATH),
    "codebook_sheet": CODEBOOK_SHEET,
    "examples_workbook": str(EXAMPLES_WORKBOOK_PATH),
    "examples_sheet": EXAMPLES_SHEET,
    "input_documents": len(pdf_paths),
    "logical_paragraph_occurrences": len(paragraphs_df),
    "unique_paragraph_texts": len(unique_paragraphs_df),
    "cross_page_paragraphs": int(paragraphs_df["spans_multiple_pages"].sum()),
    "cross_column_paragraphs": int(paragraphs_df["spans_multiple_columns"].sum()),
    "fewshot_examples_used": len(prompt_examples_df),
    "all_extraction_checks_passed": all_extraction_checks_passed,
    "extraction_approved": EXTRACTION_APPROVED,
    "mock_mode": MOCK_MODE,
}])
model_manifest_df = pd.DataFrame(MODEL_CONFIGS)

export_frames = [
    ("Logical Paragraphs", paragraphs_df),
    ("Extraction Manifest", extraction_manifest_df),
    ("Extraction Checks", extraction_quality_df),
    ("Extractor Alignment", extractor_alignment_df),
    ("Page Agreement", page_extractor_agreement_df),
    ("Line Audit", line_audit_df),
    ("Few-shot Examples", prompt_examples_df),
    ("Example Leakage Audit", example_leakage_audit_df),
    ("Model Predictions Long", model_predictions_long_df),
    ("Model Predictions Wide", model_predictions_wide_df),
    ("Consensus", consensus_annotations_df),
    ("Reliability Overall", reliability_overall_df),
    ("Reliability Pairwise", reliability_pairwise_df),
    ("Reliability by Doc", reliability_by_document_df),
    ("Run Manifest", run_manifest_df),
    ("Model Manifest", model_manifest_df),
    ("Codebook Used", codebook_df),
]

with pd.ExcelWriter(RESULTS_WORKBOOK_PATH, engine="xlsxwriter") as writer:
    for sheet_name, frame in export_frames:
        frame.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    workbook = writer.book
    header_format = workbook.add_format({
        "bold": True, "font_color": "white", "bg_color": "#1F4E78",
        "border": 1, "text_wrap": True, "valign": "top",
    })
    yes_format = workbook.add_format({"bg_color": "#C6EFCE", "font_color": "#006100"})
    disagreement_format = workbook.add_format({"bg_color": "#FFF2CC", "font_color": "#7F6000"})
    for sheet_name, frame in export_frames:
        actual_name = sheet_name[:31]
        ws = writer.sheets[actual_name]
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, max(len(frame), 1), max(len(frame.columns)-1, 0))
        for col_idx, column in enumerate(frame.columns):
            ws.write(0, col_idx, column, header_format)
            sample = [len(str(column))] + [len(str(x)) for x in frame[column].head(100).fillna("")]
            width = min(max(sample) + 2, 45)
            if column.lower() in {"text", "rationale", "consensus_rationale", "segments_json"} or "rationale" in column.lower():
                width = 45
            ws.set_column(col_idx, col_idx, max(10, width))
        if actual_name == "Consensus" and not frame.empty:
            consensus_col = frame.columns.get_loc("consensus_unlearning")
            bucket_col = frame.columns.get_loc("agreement_bucket")
            ws.conditional_format(1, consensus_col, len(frame), consensus_col,
                                  {"type": "cell", "criteria": "==", "value": True, "format": yes_format})
            ws.conditional_format(1, bucket_col, len(frame), bucket_col,
                                  {"type": "text", "criteria": "containing", "value": "majority", "format": disagreement_format})

print("Saved:", RESULTS_WORKBOOK_PATH)

Saved: /content/unlearning_pipeline/outputs/unlearning_annotation_results.xlsx


## 15. Add editable highlights and popup comments to each source PDF

In [27]:
ANNOTATION_COLORS = {
    "unanimous_yes": (0.35, 0.85, 0.35),
    "majority_yes": (1.00, 0.82, 0.20),
    "majority_no_one_positive": (1.00, 0.45, 0.45),
    "unanimous_no": (0.55, 0.75, 1.00),
}


def yn(value) -> str:
    return "Yes" if bool(value) else "No"


def model_comment_line(row: pd.Series, provider: str) -> list[str]:
    label = yn(row[f"{provider}_unlearning"])
    target = TARGET_DISPLAY.get(row[f"{provider}_target_type"], row[f"{provider}_target_type"])
    confidence = row.get(f"{provider}_confidence", np.nan)
    confidence_text = f"{float(confidence):.0%}" if pd.notna(confidence) else "n/a"
    agency = clean_text(row.get(f"{provider}_agency", "")) or "Not stated"
    rationale = clean_text(row.get(f"{provider}_rationale", ""))
    return [
        f"{provider.upper()} - {row.get(f'{provider}_model', '')} - {row.get(f'{provider}_strategy', '')}",
        f"Label: {label} | Target: {target} | Confidence: {confidence_text} | Agency: {agency}",
        f"Rationale: {rationale}",
    ]


def build_annotation_comment(row: pd.Series) -> str:
    consensus_conf = row.get("consensus_confidence", np.nan)
    conf_text = f"{float(consensus_conf):.0%}" if pd.notna(consensus_conf) else "n/a"
    lines = [
        f"Paragraph ID: {row['paragraph_id']}",
        f"Pages: {row['page_numbers']}",
        f"Consensus: {yn(row['consensus_unlearning'])} ({row['yes_votes']}/3 Yes votes)",
        f"Agreement: {row['agreement_bucket']}",
        f"Consensus target: {row['consensus_target_display']}",
        f"Consensus agency/agencies: {clean_text(row.get('consensus_agencies', '')) or 'Not stated'}",
        f"Consensus confidence: {conf_text}",
        f"Review priority: {row['review_priority']}", "",
    ]
    for provider in PROVIDER_ORDER:
        lines.extend(model_comment_line(row, provider)); lines.append("")
    return "\n".join(lines)[:MAX_COMMENT_CHARS]


def should_annotate(row: pd.Series) -> bool:
    if row["agreement_bucket"] == "unanimous_no":
        return ANNOTATE_UNANIMOUS_NO
    return row["yes_votes"] >= 1 if ANNOTATE_ANY_POSITIVE_VOTE else bool(row["consensus_unlearning"])


def grouped_page_segments(segments_json: str) -> dict[int, dict]:
    grouped = defaultdict(lambda: {"rects": [], "texts": []})
    for segment in json.loads(segments_json):
        page_number = int(segment["page_number"])
        grouped[page_number]["rects"].extend(segment["rects"])
        grouped[page_number]["texts"].append(segment.get("text", ""))
    return dict(grouped)


def existing_annotation_count(doc: fitz.Document) -> int:
    total = 0
    for page in doc:
        annot = page.first_annot
        while annot:
            total += 1; annot = annot.next
    return total


def annotate_pdf(source_path: Path, annotations: pd.DataFrame, output_path: Path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    expected_rows, failures = [], []
    with fitz.open(source_path) as doc:
        existing = existing_annotation_count(doc)
        if existing and not STRIP_EXISTING_ANNOTATIONS:
            raise RuntimeError(f"{source_path.name} already contains {existing} annotations. Enable stripping or use a clean source PDF.")
        if existing:
            for page in doc:
                annot = page.first_annot
                while annot:
                    nxt = annot.next; page.delete_annot(annot); annot = nxt

        for _, row in annotations.sort_values(["paragraph_order"]).iterrows():
            if not should_annotate(row):
                continue
            for page_number, segment in grouped_page_segments(row["segments_json"]).items():
                key = f"{row['paragraph_id']}|p{page_number}"
                rects = [fitz.Rect(*values) for values in segment["rects"]]
                expected_rows.append({
                    "source_file": str(source_path), "paragraph_id": row["paragraph_id"],
                    "page_number": page_number, "annotation_key": key,
                    "expected_segment_text": clean_joined_text(" ".join(segment["texts"])),
                    "rects_json": json.dumps(segment["rects"]),
                })
                try:
                    page = doc[page_number - 1]
                    annot = page.add_highlight_annot(rects)
                    annot.set_colors(stroke=ANNOTATION_COLORS[row["agreement_bucket"]])
                    annot.set_opacity(0.38)
                    annot.set_info(
                        title="Unlearning Model Ensemble",
                        subject=key,
                        content=build_annotation_comment(row),
                    )
                    annot.update()
                except Exception as exc:
                    failures.append(f"{key}: {exc}")
        doc.save(output_path, garbage=4, deflate=True)

    return {
        "source_pdf": str(source_path), "annotated_pdf": str(output_path),
        "existing_annotations_removed": existing,
        "expected_annotations": len(expected_rows),
        "annotation_failures": len(failures),
        "failure_details": " | ".join(failures[:20]),
    }, expected_rows


annotation_audit_rows, expected_annotation_rows = [], []
for source_file, subset in consensus_annotations_df.groupby("source_file"):
    source_path = Path(source_file)
    output_path = ANNOTATED_DIR / f"{source_path.stem}_unlearning_annotated_v2.pdf"
    audit, expected = annotate_pdf(source_path, subset, output_path)
    annotation_audit_rows.append(audit); expected_annotation_rows.extend(expected)

annotation_audit_df = pd.DataFrame(annotation_audit_rows)
expected_annotations_df = pd.DataFrame(expected_annotation_rows)
annotation_audit_df.to_csv(OUTPUT_DIR / "annotation_audit.csv", index=False)
expected_annotations_df.to_csv(OUTPUT_DIR / "expected_annotation_segments.csv", index=False)
display(annotation_audit_df)
if annotation_audit_df["annotation_failures"].sum() > 0:
    raise RuntimeError("Some annotations failed. Review annotation_audit.csv.")

,source_pdf,annotated_pdf,existing_annotations_removed,expected_annotations,annotation_failures,failure_details
0,/content/unlearning_pipeline/input_documents/Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,/content/unlearning_pipeline/outputs/annotated_pdfs/Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of_...,0,18,0,


## 16. ID-level, geometry-level, and text-level verification

In [28]:
def annotation_records(pdf_path: Path) -> pd.DataFrame:
    rows = []
    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            annot = page.first_annot
            while annot:
                info = annot.info or {}
                subject = clean_text(info.get("subject", ""))
                paragraph_id = subject.split("|p", 1)[0] if "|p" in subject else ""
                rows.append({
                    "annotated_pdf": str(pdf_path), "page_number": page_number,
                    "annotation_key": subject, "paragraph_id": paragraph_id,
                    "annotation_rect": json.dumps(list(annot.rect)),
                })
                annot = annot.next
    return pd.DataFrame(rows)


def text_from_rects(page: fitz.Page, rect_values: list[list[float]]) -> str:
    parts = [page.get_textbox(fitz.Rect(*values)) for values in rect_values]
    return clean_joined_text(" ".join(parts))


verification_detail_rows = []
for _, audit in annotation_audit_df.iterrows():
    annotated_path = Path(audit["annotated_pdf"])
    expected = expected_annotations_df[expected_annotations_df["source_file"].eq(audit["source_pdf"])].copy()
    actual = annotation_records(annotated_path)
    expected_keys, actual_keys = set(expected["annotation_key"]), set(actual["annotation_key"])

    with fitz.open(annotated_path) as doc:
        for _, row in expected.iterrows():
            rect_values = json.loads(row["rects_json"])
            page = doc[int(row["page_number"]) - 1]
            extracted = text_from_rects(page, rect_values)
            expected_text = clean_joined_text(row["expected_segment_text"])
            similarity = SequenceMatcher(
                None, normalize_for_match(expected_text), normalize_for_match(extracted)
            ).ratio()
            key_actual = actual[actual["annotation_key"].eq(row["annotation_key"])]
            annotation_found = len(key_actual) == 1
            geometry_ok = False
            if annotation_found:
                expected_union = fitz.Rect(*rect_values[0])
                for values in rect_values[1:]: expected_union |= fitz.Rect(*values)
                annotation_rect = fitz.Rect(*json.loads(key_actual.iloc[0]["annotation_rect"]))
                intersection = expected_union & annotation_rect
                geometry_ok = (intersection.get_area() / max(expected_union.get_area(), 1e-9)) >= 0.95
            verification_detail_rows.append({
                "annotated_pdf": str(annotated_path), "annotation_key": row["annotation_key"],
                "paragraph_id": row["paragraph_id"], "page_number": row["page_number"],
                "annotation_found_once": annotation_found, "geometry_covers_expected": geometry_ok,
                "coordinate_text_similarity": similarity,
                "text_similarity_passed": similarity >= MIN_ANNOTATION_TEXT_SIMILARITY,
            })

    if expected_keys != actual_keys:
        missing = sorted(expected_keys - actual_keys)[:20]
        extra = sorted(actual_keys - expected_keys)[:20]
        raise RuntimeError(f"Annotation ID verification failed for {annotated_path.name}. Missing={missing}; Extra={extra}")

verification_detail_df = pd.DataFrame(verification_detail_rows)
verification_summary_df = verification_detail_df.groupby("annotated_pdf", as_index=False).agg(
    expected_annotations=("annotation_key", "size"),
    annotation_ids_matched=("annotation_found_once", "sum"),
    geometry_checks_passed=("geometry_covers_expected", "sum"),
    text_checks_passed=("text_similarity_passed", "sum"),
    minimum_text_similarity=("coordinate_text_similarity", "min"),
)
verification_summary_df["all_checks_passed"] = (
    verification_summary_df["expected_annotations"].eq(verification_summary_df["annotation_ids_matched"])
    & verification_summary_df["expected_annotations"].eq(verification_summary_df["geometry_checks_passed"])
    & verification_summary_df["expected_annotations"].eq(verification_summary_df["text_checks_passed"])
)
verification_detail_df.to_csv(OUTPUT_DIR / "annotation_verification_detail.csv", index=False)
verification_summary_df.to_csv(OUTPUT_DIR / "annotation_verification_summary.csv", index=False)

if not verification_summary_df["all_checks_passed"].all():
    display(verification_detail_df[
        ~(verification_detail_df["annotation_found_once"]
          & verification_detail_df["geometry_covers_expected"]
          & verification_detail_df["text_similarity_passed"])
    ])
    raise RuntimeError("Annotation verification failed.")

expected_prediction_rows = len(unique_paragraphs_df) * len(MODEL_CONFIGS)
actual_prediction_rows = len(valid_unique_predictions_df)
if actual_prediction_rows != expected_prediction_rows:
    raise RuntimeError(f"Expected {expected_prediction_rows} successful predictions; found {actual_prediction_rows}.")

archive_base = BASE_DIR / "unlearning_document_annotation_outputs_v2"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR))
display(verification_summary_df)
print("Completed outputs:")
print(" - Extraction review workbook:", EXTRACTION_REVIEW_WORKBOOK)
print(" - Results workbook:", RESULTS_WORKBOOK_PATH)
print(" - Extraction audit PDFs:", EXTRACTION_AUDIT_DIR)
print(" - Final annotated PDFs:", ANNOTATED_DIR)
print(" - ZIP archive:", archive_path)

,annotated_pdf,expected_annotations,annotation_ids_matched,geometry_checks_passed,text_checks_passed,minimum_text_similarity,all_checks_passed
0,/content/unlearning_pipeline/outputs/annotated_pdfs/Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of_...,18,18,18,18,0.843188,True


Completed outputs:
 - Extraction review workbook: /content/unlearning_pipeline/outputs/paragraph_extraction_review.xlsx
 - Results workbook: /content/unlearning_pipeline/outputs/unlearning_annotation_results.xlsx
 - Extraction audit PDFs: /content/unlearning_pipeline/outputs/extraction_audits
 - Final annotated PDFs: /content/unlearning_pipeline/outputs/annotated_pdfs
 - ZIP archive: /content/unlearning_pipeline/unlearning_document_annotation_outputs_v2.zip


## Interpretation notes

- **Blue extraction-audit highlights** are not model labels. They show the logical paragraph units that will be sent to all three providers.
- The final model annotation colors retain the prior scheme: green = unanimous Yes, yellow = majority Yes, red = one positive vote, and unanimous No is unmarked by default.
- Cross-page paragraphs are classified once and highlighted on every page segment using the same paragraph ID.
- Reliability is calculated on reconstructed logical paragraphs, not raw PDF blocks.